In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import vectorbt as vbt
import ta

print("Q-SCANNER READY")

Q-SCANNER READY


In [2]:
print("Testing Q-Scanner kernel...")
print("Python is working.")

Testing Q-Scanner kernel...
Python is working.


In [3]:
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("yfinance:", yf.__version__)
print("VectorBT:", vbt.__version__)
print("TA:", ta.__version__)

NumPy: 2.5.2
Pandas: 3.0.5
yfinance: 1.7.0
VectorBT: 1.1.0


AttributeError: module 'ta' has no attribute '__version__'

In [4]:
print("TA library loaded successfully")

from ta.trend import SMAIndicator
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

print("SMAIndicator: OK")
print("RSIIndicator: OK")
print("BollingerBands: OK")

TA library loaded successfully
SMAIndicator: OK
RSIIndicator: OK
BollingerBands: OK


In [5]:
# ==========================================
# Q-SCANNER — MARKET DATA TEST
# ==========================================

import yfinance as yf
import pandas as pd
import numpy as np

ticker = "AAPL"

data = yf.download(
    ticker,
    period="6mo",
    interval="1d",
    auto_adjust=True,
    progress=False
)

print("Ticker:", ticker)
print("Rows:", len(data))
print("\nLatest data:")
display(data.tail())

Ticker: AAPL
Rows: 129

Latest data:


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2026-08-31,316.850006,321.239990,312.799988,319.600006,41242700
2026-09-01,325.130005,327.299988,314.730011,316.980011,53167400
2026-09-02,324.959991,328.399994,323.529999,326.869995,33776400
2026-09-03,328.209991,330.809998,324.109985,324.869995,37203500
2026-09-04,326.279999,328.929993,326.209991,328.304993,1920787


In [6]:
# ==========================================
# Q-SCANNER — TECHNICAL INDICATORS
# ==========================================

from ta.trend import SMAIndicator
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

# Make sure we are working with a Series
close = data["Close"].squeeze()

# Moving averages
data["SMA_20"] = SMAIndicator(
    close=close,
    window=20
).sma_indicator()

data["SMA_50"] = SMAIndicator(
    close=close,
    window=50
).sma_indicator()

# RSI
data["RSI"] = RSIIndicator(
    close=close,
    window=14
).rsi()

# Bollinger Bands
bb = BollingerBands(
    close=close,
    window=20,
    window_dev=2
)

data["BB_Upper"] = bb.bollinger_hband()
data["BB_Middle"] = bb.bollinger_mavg()
data["BB_Lower"] = bb.bollinger_lband()

print("Technical indicators calculated successfully.")

display(
    data[
        [
            "Close",
            "SMA_20",
            "SMA_50",
            "RSI",
            "BB_Upper",
            "BB_Middle",
            "BB_Lower"
        ]
    ].tail(10)
)

Technical indicators calculated successfully.


Price,Close,SMA_20,SMA_50,RSI,BB_Upper,BB_Middle,BB_Lower
Ticker,AAPL,,,,,,
Date,,,,,,,
2026-08-24,310.339996,312.886340,310.293065,48.348692,334.309812,312.886340,291.462869
2026-08-25,309.899994,311.391994,310.673482,47.934362,328.915925,311.391994,293.868062
2026-08-26,313.450012,310.169565,311.019191,51.542492,322.875391,310.169565,297.463739
2026-08-27,314.579987,309.241431,311.331147,52.666887,316.746757,309.241431,301.736104
2026-08-28,319.700012,309.794240,311.811248,57.481126,318.564217,309.794240,301.024263
2026-08-31,316.850006,310.478812,312.193184,54.177866,319.207402,310.478812,301.750221
2026-09-01,325.130005,311.279642,312.760703,61.161059,322.058398,311.279642,300.500885
2026-09-02,324.959991,311.991040,313.378975,60.955645,324.300697,311.991040,299.681383


In [7]:
# ==========================================
# Q-SCANNER — SIGNAL ENGINE
# ==========================================

latest = data.iloc[-1]

# Convert single-value Series to normal numbers
price = float(np.asarray(latest["Close"]).squeeze())
sma20 = float(np.asarray(latest["SMA_20"]).squeeze())
sma50 = float(np.asarray(latest["SMA_50"]).squeeze())
rsi = float(np.asarray(latest["RSI"]).squeeze())

# Signal rules
if price > sma20 and sma20 > sma50 and 50 < rsi < 70:
    signal = "BUY"

elif price < sma20 and sma20 < sma50 and 30 < rsi < 50:
    signal = "SELL"

else:
    signal = "NEUTRAL"

print("================================")
print("        Q-SCANNER SIGNAL")
print("================================")
print(f"Price:       {price:.2f}")
print(f"SMA 20:      {sma20:.2f}")
print(f"SMA 50:      {sma50:.2f}")
print(f"RSI:         {rsi:.2f}")
print("--------------------------------")
print(f"SIGNAL:      {signal}")
print("================================")

        Q-SCANNER SIGNAL
Price:       326.28
SMA 20:      313.46
SMA 50:      315.11
RSI:         60.96
--------------------------------
SIGNAL:      NEUTRAL


In [8]:
# ==========================================
# Q-SCANNER — SCORING ENGINE
# ==========================================

score = 0
max_score = 6

# --------------------------
# 1. TREND
# --------------------------

if price > sma20:
    score += 1
    trend_price = "Bullish"
else:
    trend_price = "Bearish"

if sma20 > sma50:
    score += 1
    trend_ma = "Bullish"
else:
    trend_ma = "Bearish"


# --------------------------
# 2. MOMENTUM — RSI
# --------------------------

if 50 <= rsi < 70:
    score += 2
    momentum = "Strong"
elif rsi >= 70:
    momentum = "Overbought"
elif 30 <= rsi < 50:
    momentum = "Weak"
else:
    momentum = "Oversold"


# --------------------------
# 3. BOLLINGER BANDS
# --------------------------

bb_upper = float(np.asarray(latest["BB_Upper"]).squeeze())
bb_lower = float(np.asarray(latest["BB_Lower"]).squeeze())

if price < bb_lower:
    score += 1
    volatility_signal = "Potentially Oversold"

elif price > bb_upper:
    volatility_signal = "Potentially Overbought"

else:
    score += 1
    volatility_signal = "Normal"


# --------------------------
# 4. FINAL CLASSIFICATION
# --------------------------

if score >= 5:
    final_signal = "STRONG BUY"

elif score >= 4:
    final_signal = "BUY"

elif score <= 1:
    final_signal = "SELL"

else:
    final_signal = "NEUTRAL"


# --------------------------
# DISPLAY RESULT
# --------------------------

print("========================================")
print("          Q-SCANNER ANALYSIS")
print("========================================")
print(f"Price:              {price:.2f}")
print(f"SMA 20:             {sma20:.2f}")
print(f"SMA 50:             {sma50:.2f}")
print(f"RSI:                {rsi:.2f}")
print("----------------------------------------")
print(f"Price vs SMA20:     {trend_price}")
print(f"SMA20 vs SMA50:     {trend_ma}")
print(f"Momentum:           {momentum}")
print(f"Bollinger Signal:   {volatility_signal}")
print("----------------------------------------")
print(f"SCORE:              {score}/{max_score}")
print(f"FINAL SIGNAL:       {final_signal}")
print("========================================")

          Q-SCANNER ANALYSIS
Price:              326.28
SMA 20:             313.46
SMA 50:             315.11
RSI:                60.96
----------------------------------------
Price vs SMA20:     Bullish
SMA20 vs SMA50:     Bearish
Momentum:           Strong
Bollinger Signal:   Normal
----------------------------------------
SCORE:              4/6
FINAL SIGNAL:       BUY


In [9]:
# ============================================================
# Q-SCANNER — MULTI-ASSET SCANNER
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import ta

# Assets to scan
tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META",
    "GOOGL",
    "TSLA"
]

results = []

for ticker in tickers:

    try:
        # Download market data
        data = yf.download(
            ticker,
            period="6mo",
            interval="1d",
            auto_adjust=True,
            progress=False
        )

        # Skip if no data
        if data.empty:
            print(f"{ticker}: No data")
            continue

        # Handle possible MultiIndex columns
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        # Calculate indicators
        data["SMA_20"] = ta.trend.SMAIndicator(
            close=data["Close"],
            window=20
        ).sma_indicator()

        data["SMA_50"] = ta.trend.SMAIndicator(
            close=data["Close"],
            window=50
        ).sma_indicator()

        data["RSI"] = ta.momentum.RSIIndicator(
            close=data["Close"],
            window=14
        ).rsi()

        bb = ta.volatility.BollingerBands(
            close=data["Close"],
            window=20,
            window_dev=2
        )

        data["BB_Upper"] = bb.bollinger_hband()
        data["BB_Middle"] = bb.bollinger_mavg()
        data["BB_Lower"] = bb.bollinger_lband()

        # Remove incomplete rows
        data = data.dropna()

        if data.empty:
            continue

        latest = data.iloc[-1]

        # Convert values to normal numbers
        price = float(np.asarray(latest["Close"]).squeeze())
        sma20 = float(np.asarray(latest["SMA_20"]).squeeze())
        sma50 = float(np.asarray(latest["SMA_50"]).squeeze())
        rsi = float(np.asarray(latest["RSI"]).squeeze())
        bb_upper = float(np.asarray(latest["BB_Upper"]).squeeze())
        bb_lower = float(np.asarray(latest["BB_Lower"]).squeeze())

        # ====================================================
        # SIGNAL SCORING
        # ====================================================

        score = 0
        max_score = 6

        # 1. Price vs SMA20
        if price > sma20:
            trend_price = "Bullish"
            score += 1
        else:
            trend_price = "Bearish"

        # 2. SMA20 vs SMA50
        if sma20 > sma50:
            trend_ma = "Bullish"
            score += 1
        else:
            trend_ma = "Bearish"

        # 3. RSI / Momentum
        if 50 < rsi < 70:
            momentum = "Strong"
            score += 1
        elif 30 < rsi <= 50:
            momentum = "Weak"
        elif rsi >= 70:
            momentum = "Overbought"
        else:
            momentum = "Oversold"
            score += 1

        # 4. Bollinger Bands
        if price < bb_lower:
            volatility_signal = "Potentially Oversold"
            score += 2
        elif price > bb_upper:
            volatility_signal = "Potentially Overbought"
        else:
            volatility_signal = "Normal"

        # 5. Final signal
        if score >= 5:
            final_signal = "STRONG BUY"
        elif score == 4:
            final_signal = "BUY"
        elif score <= 1:
            final_signal = "SELL"
        else:
            final_signal = "NEUTRAL"

        # Store results
        results.append({
            "Ticker": ticker,
            "Price": round(price, 2),
            "SMA_20": round(sma20, 2),
            "SMA_50": round(sma50, 2),
            "RSI": round(rsi, 2),
            "Price Trend": trend_price,
            "MA Trend": trend_ma,
            "Momentum": momentum,
            "Bollinger": volatility_signal,
            "Score": score,
            "Signal": final_signal
        })

    except Exception as e:
        print(f"{ticker}: ERROR -> {e}")


# ============================================================
# CREATE SCANNER TABLE
# ============================================================

scanner_results = pd.DataFrame(results)

if not scanner_results.empty:

    # Rank strongest signals first
    scanner_results = scanner_results.sort_values(
        by=["Score", "RSI"],
        ascending=[False, False]
    ).reset_index(drop=True)

    print("=" * 100)
    print("                 Q-SCANNER — MULTI-ASSET ANALYSIS")
    print("=" * 100)

    display(scanner_results)

else:
    print("No assets were successfully scanned.")

                 Q-SCANNER — MULTI-ASSET ANALYSIS


,Ticker,Price,SMA_20,SMA_50,RSI,Price Trend,MA Trend,Momentum,Bollinger,Score,Signal
0,MSFT,505.82,495.65,443.68,63.42,Bullish,Bullish,Strong,Normal,3,NEUTRAL
1,NVDA,232.34,220.18,210.61,61.55,Bullish,Bullish,Strong,Normal,3,NEUTRAL
2,AAPL,326.36,313.46,315.12,61.06,Bullish,Bearish,Strong,Normal,2,NEUTRAL
3,META,610.73,576.58,595.37,60.71,Bullish,Bearish,Strong,Normal,2,NEUTRAL
4,TSLA,360.68,348.77,358.08,53.44,Bullish,Bearish,Strong,Normal,2,NEUTRAL
5,AMZN,259.28,262.24,253.96,50.24,Bearish,Bullish,Strong,Normal,2,NEUTRAL
6,GOOGL,341.74,343.77,348.63,47.20,Bearish,Bearish,Weak,Normal,0,SELL


In [10]:
# ============================================================
# Q-SCANNER — IMPROVED SIGNAL ENGINE v2
# ============================================================

results_v2 = []

for ticker in tickers:

    try:
        # Download market data
        data = yf.download(
            ticker,
            period="6mo",
            interval="1d",
            auto_adjust=True,
            progress=False
        )

        if data.empty:
            continue

        # Handle MultiIndex columns
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        # ----------------------------------------------------
        # TECHNICAL INDICATORS
        # ----------------------------------------------------

        data["SMA_20"] = ta.trend.SMAIndicator(
            close=data["Close"],
            window=20
        ).sma_indicator()

        data["SMA_50"] = ta.trend.SMAIndicator(
            close=data["Close"],
            window=50
        ).sma_indicator()

        data["RSI"] = ta.momentum.RSIIndicator(
            close=data["Close"],
            window=14
        ).rsi()

        bb = ta.volatility.BollingerBands(
            close=data["Close"],
            window=20,
            window_dev=2
        )

        data["BB_Upper"] = bb.bollinger_hband()
        data["BB_Middle"] = bb.bollinger_mavg()
        data["BB_Lower"] = bb.bollinger_lband()

        data = data.dropna()

        if data.empty:
            continue

        latest = data.iloc[-1]

        # Convert values to numbers
        price = float(np.asarray(latest["Close"]).squeeze())
        sma20 = float(np.asarray(latest["SMA_20"]).squeeze())
        sma50 = float(np.asarray(latest["SMA_50"]).squeeze())
        rsi = float(np.asarray(latest["RSI"]).squeeze())
        bb_upper = float(np.asarray(latest["BB_Upper"]).squeeze())
        bb_lower = float(np.asarray(latest["BB_Lower"]).squeeze())

        # ====================================================
        # IMPROVED SCORING
        # Maximum = +6
        # Minimum = -6
        # ====================================================

        score = 0

        # ----------------------------------------------------
        # 1. PRICE VS SMA20
        # ----------------------------------------------------

        if price > sma20:
            price_trend = "Bullish"
            score += 1
        else:
            price_trend = "Bearish"
            score -= 1

        # ----------------------------------------------------
        # 2. SMA20 VS SMA50
        # ----------------------------------------------------

        if sma20 > sma50:
            moving_average_trend = "Bullish"
            score += 1
        else:
            moving_average_trend = "Bearish"
            score -= 1

        # ----------------------------------------------------
        # 3. RSI / MOMENTUM
        # ----------------------------------------------------

        if 50 <= rsi < 70:
            momentum = "Bullish"
            score += 1

        elif 70 <= rsi:
            momentum = "Overbought"
            score -= 1

        elif 30 < rsi < 50:
            momentum = "Bearish"
            score -= 1

        else:
            momentum = "Oversold"
            score += 1

        # ----------------------------------------------------
        # 4. BOLLINGER BANDS
        # ----------------------------------------------------

        if price < bb_lower:
            bollinger_signal = "Oversold"
            score += 2

        elif price > bb_upper:
            bollinger_signal = "Overbought"
            score -= 2

        else:
            bollinger_signal = "Normal"

        # ----------------------------------------------------
        # 5. FINAL SIGNAL
        # ----------------------------------------------------

        if score >= 4:
            final_signal = "STRONG BUY"

        elif score >= 2:
            final_signal = "BUY"

        elif score <= -4:
            final_signal = "STRONG SELL"

        elif score <= -2:
            final_signal = "SELL"

        else:
            final_signal = "NEUTRAL"

        # ----------------------------------------------------
        # CONFIDENCE
        # ----------------------------------------------------

        confidence = round((abs(score) / 6) * 100, 1)

        # ----------------------------------------------------
        # SAVE RESULT
        # ----------------------------------------------------

        results_v2.append({
            "Ticker": ticker,
            "Price": round(price, 2),
            "SMA_20": round(sma20, 2),
            "SMA_50": round(sma50, 2),
            "RSI": round(rsi, 2),
            "Price Trend": price_trend,
            "MA Trend": moving_average_trend,
            "Momentum": momentum,
            "Bollinger": bollinger_signal,
            "Score": score,
            "Confidence %": confidence,
            "Signal": final_signal
        })

    except Exception as e:
        print(f"{ticker}: ERROR -> {e}")


# ============================================================
# CREATE IMPROVED SCANNER TABLE
# ============================================================

scanner_v2 = pd.DataFrame(results_v2)

if not scanner_v2.empty:

    # Strongest signals first
    scanner_v2["Signal Rank"] = scanner_v2["Score"].abs()

    scanner_v2 = scanner_v2.sort_values(
        by=["Score", "RSI"],
        ascending=[False, False]
    ).reset_index(drop=True)

    # Remove helper column
    scanner_v2 = scanner_v2.drop(columns=["Signal Rank"])

    print("=" * 110)
    print("              Q-SCANNER — IMPROVED MULTI-ASSET ANALYSIS")
    print("=" * 110)

    display(scanner_v2)

else:
    print("No assets were successfully scanned.")

              Q-SCANNER — IMPROVED MULTI-ASSET ANALYSIS


,Ticker,Price,SMA_20,SMA_50,RSI,Price Trend,MA Trend,Momentum,Bollinger,Score,Confidence %,Signal
0,MSFT,505.98,495.65,443.69,63.53,Bullish,Bullish,Bullish,Normal,3,50.0,BUY
1,NVDA,232.32,220.18,210.61,61.54,Bullish,Bullish,Bullish,Normal,3,50.0,BUY
2,AAPL,326.64,313.47,315.12,61.42,Bullish,Bearish,Bullish,Normal,1,16.7,NEUTRAL
3,META,610.73,576.58,595.37,60.71,Bullish,Bearish,Bullish,Normal,1,16.7,NEUTRAL
4,TSLA,359.98,348.73,358.07,53.16,Bullish,Bearish,Bullish,Normal,1,16.7,NEUTRAL
5,AMZN,259.41,262.24,253.97,50.35,Bearish,Bullish,Bullish,Normal,1,16.7,NEUTRAL
6,GOOGL,341.69,343.77,348.63,47.16,Bearish,Bearish,Bearish,Normal,-3,50.0,SELL


In [11]:
# ============================================================
# Q-SCANNER — SIGNAL ENGINE V3
# ============================================================

def generate_signal(row):

    score = 0
    reasons = []

    # --------------------------------------------------------
    # 1. PRICE VS SMA 20
    # --------------------------------------------------------
    if row["Price"] > row["SMA_20"]:
        score += 1
        reasons.append("Price above SMA20")
    else:
        score -= 1
        reasons.append("Price below SMA20")

    # --------------------------------------------------------
    # 2. SMA 20 VS SMA 50
    # --------------------------------------------------------
    if row["SMA_20"] > row["SMA_50"]:
        score += 1
        reasons.append("SMA20 above SMA50")
    else:
        score -= 1
        reasons.append("SMA20 below SMA50")

    # --------------------------------------------------------
    # 3. RSI
    # --------------------------------------------------------
    rsi = float(row["RSI"])

    if 50 <= rsi < 65:
        score += 1
        reasons.append("Healthy bullish RSI")

    elif 65 <= rsi < 70:
        score += 0.5
        reasons.append("Strong RSI")

    elif 30 < rsi < 50:
        score -= 1
        reasons.append("Bearish RSI")

    elif rsi <= 30:
        score += 0.5
        reasons.append("Potential RSI reversal")

    elif rsi >= 70:
        score -= 1
        reasons.append("RSI overbought")

    # --------------------------------------------------------
    # 4. MOMENTUM
    # --------------------------------------------------------
    momentum = str(row["Momentum"])

    if momentum == "Bullish":
        score += 1
        reasons.append("Bullish momentum")

    elif momentum == "Bearish":
        score -= 1
        reasons.append("Bearish momentum")

    # --------------------------------------------------------
    # 5. BOLLINGER
    # --------------------------------------------------------
    bollinger = str(row["Bollinger"])

    if bollinger == "Potentially Overbought":
        score -= 0.5
        reasons.append("Potentially overbought")

    elif bollinger == "Potentially Oversold":
        score += 0.5
        reasons.append("Potentially oversold")

    # --------------------------------------------------------
    # FINAL SIGNAL
    # --------------------------------------------------------
    if score >= 3:
        signal = "STRONG BUY"

    elif score >= 2:
        signal = "BUY"

    elif score <= -3:
        signal = "STRONG SELL"

    elif score <= -2:
        signal = "SELL"

    else:
        signal = "NEUTRAL"

    # --------------------------------------------------------
    # CONFIDENCE
    # --------------------------------------------------------
    max_score = 5.5
    confidence = min(abs(score) / max_score * 100, 100)

    return pd.Series({
        "New Score": round(score, 2),
        "Confidence %": round(confidence, 1),
        "New Signal": signal,
        "Reasons": "; ".join(reasons)
    })


# ============================================================
# CREATE V3 TABLE
# ============================================================

# Remove old scoring columns if they already exist
base = scanner_v2.copy()

for column in ["Score", "Confidence %", "Signal", "Reasons", "Signal Rank"]:
    if column in base.columns:
        base = base.drop(columns=[column])


# Generate new signals
signal_output = base.apply(generate_signal, axis=1)

# Combine everything
scanner_v3 = pd.concat(
    [base, signal_output],
    axis=1
)


# ============================================================
# RANK STRONGEST SIGNALS
# ============================================================

scanner_v3["Signal Rank"] = scanner_v3["New Score"].abs()

scanner_v3 = scanner_v3.sort_values(
    by=["Signal Rank", "Confidence %"],
    ascending=[False, False]
).reset_index(drop=True)


# Remove helper column
scanner_v3 = scanner_v3.drop(columns=["Signal Rank"])


# ============================================================
# DISPLAY
# ============================================================

print("=" * 120)
print("Q-SCANNER — SIGNAL ENGINE V3")
print("=" * 120)

display(scanner_v3)

Q-SCANNER — SIGNAL ENGINE V3


,Ticker,Price,SMA_20,SMA_50,RSI,Price Trend,MA Trend,Momentum,Bollinger,New Score,Confidence %,New Signal,Reasons
0,MSFT,505.98,495.65,443.69,63.53,Bullish,Bullish,Bullish,Normal,4,72.7,STRONG BUY,Price above SMA20; SMA20 above SMA50; Healthy ...
1,NVDA,232.32,220.18,210.61,61.54,Bullish,Bullish,Bullish,Normal,4,72.7,STRONG BUY,Price above SMA20; SMA20 above SMA50; Healthy ...
2,GOOGL,341.69,343.77,348.63,47.16,Bearish,Bearish,Bearish,Normal,-4,72.7,STRONG SELL,Price below SMA20; SMA20 below SMA50; Bearish ...
3,AAPL,326.64,313.47,315.12,61.42,Bullish,Bearish,Bullish,Normal,2,36.4,BUY,Price above SMA20; SMA20 below SMA50; Healthy ...
4,META,610.73,576.58,595.37,60.71,Bullish,Bearish,Bullish,Normal,2,36.4,BUY,Price above SMA20; SMA20 below SMA50; Healthy ...
5,TSLA,359.98,348.73,358.07,53.16,Bullish,Bearish,Bullish,Normal,2,36.4,BUY,Price above SMA20; SMA20 below SMA50; Healthy ...
6,AMZN,259.41,262.24,253.97,50.35,Bearish,Bullish,Bullish,Normal,2,36.4,BUY,Price below SMA20; SMA20 above SMA50; Healthy ...


In [12]:
# ============================================================
# Q-SCANNER — HISTORICAL BACKTEST ENGINE V1
# ============================================================

import yfinance as yf
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# ASSETS TO BACKTEST
# ------------------------------------------------------------

tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META",
    "GOOGL",
    "TSLA"
]


# ------------------------------------------------------------
# BACKTEST SETTINGS
# ------------------------------------------------------------

period = "5y"

transaction_cost = 0.001   # 0.10% per trade


# ------------------------------------------------------------
# FUNCTION: CALCULATE INDICATORS
# ------------------------------------------------------------

def calculate_indicators(df):

    df = df.copy()

    # Simple Moving Averages
    df["SMA_20"] = df["Close"].rolling(20).mean()
    df["SMA_50"] = df["Close"].rolling(50).mean()

    # RSI
    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["RSI"] = 100 - (100 / (1 + rs))

    # Bollinger Bands
    df["BB_Middle"] = df["Close"].rolling(20).mean()
    df["BB_STD"] = df["Close"].rolling(20).std()

    df["BB_Upper"] = (
        df["BB_Middle"] + 2 * df["BB_STD"]
    )

    df["BB_Lower"] = (
        df["BB_Middle"] - 2 * df["BB_STD"]
    )

    # Momentum
    df["Momentum_Value"] = df["Close"].pct_change(10)

    return df


# ------------------------------------------------------------
# FUNCTION: GENERATE SCORE
# ------------------------------------------------------------

def calculate_score(row):

    score = 0

    price = row["Close"]
    sma20 = row["SMA_20"]
    sma50 = row["SMA_50"]
    rsi = row["RSI"]

    # Price vs SMA20
    if price > sma20:
        score += 1
    else:
        score -= 1

    # SMA20 vs SMA50
    if sma20 > sma50:
        score += 1
    else:
        score -= 1

    # RSI
    if 50 <= rsi < 65:
        score += 1

    elif 65 <= rsi < 70:
        score += 0.5

    elif 30 < rsi < 50:
        score -= 1

    elif rsi <= 30:
        score += 0.5

    elif rsi >= 70:
        score -= 1

    # Momentum
    if row["Momentum_Value"] > 0:
        score += 1
    else:
        score -= 1

    # Bollinger condition
    if price > row["BB_Upper"]:
        score -= 0.5

    elif price < row["BB_Lower"]:
        score += 0.5

    return score


# ------------------------------------------------------------
# BACKTEST ONE ASSET
# ------------------------------------------------------------

def backtest_ticker(ticker):

    print(f"Backtesting {ticker}...")

    df = yf.download(
        ticker,
        period=period,
        interval="1d",
        auto_adjust=True,
        progress=False
    )

    if df.empty:
        return None

    # Handle yfinance multi-level columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = calculate_indicators(df)

    # Remove incomplete rows
    df = df.dropna().copy()

    # Calculate today's score
    df["Score"] = df.apply(calculate_score, axis=1)

    # --------------------------------------------------------
    # SIGNAL
    # --------------------------------------------------------

    df["Signal"] = "NEUTRAL"

    df.loc[df["Score"] >= 2, "Signal"] = "BUY"
    df.loc[df["Score"] <= -2, "Signal"] = "SELL"

    # --------------------------------------------------------
    # POSITION
    # --------------------------------------------------------

    df["Position"] = 0

    df.loc[df["Score"] >= 2, "Position"] = 1
    df.loc[df["Score"] <= -2, "Position"] = -1

    # --------------------------------------------------------
    # NEXT-DAY RETURN
    # --------------------------------------------------------

    df["Market_Return"] = df["Close"].pct_change().shift(-1)

    # Signal generated today acts on the NEXT trading day
    df["Strategy_Return"] = (
        df["Position"] * df["Market_Return"]
    )

    # --------------------------------------------------------
    # TRANSACTION COST
    # --------------------------------------------------------

    df["Position_Change"] = (
        df["Position"].diff().abs()
    )

    df["Strategy_Return"] = (
        df["Strategy_Return"]
        - df["Position_Change"] * transaction_cost
    )

    # Remove final row
    df = df.dropna(subset=["Strategy_Return"])

    # --------------------------------------------------------
    # PERFORMANCE
    # --------------------------------------------------------

    cumulative_return = (
        (1 + df["Strategy_Return"]).prod() - 1
    )

    buy_hold_return = (
        (1 + df["Market_Return"]).prod() - 1
    )

    # Winning trading days
    active_trades = df[
        df["Position"] != 0
    ]

    if len(active_trades) > 0:

        win_rate = (
            active_trades["Strategy_Return"] > 0
        ).mean() * 100

    else:
        win_rate = 0

    # Maximum drawdown
    equity_curve = (
        1 + df["Strategy_Return"]
    ).cumprod()

    running_max = equity_curve.cummax()

    drawdown = (
        equity_curve / running_max
    ) - 1

    max_drawdown = drawdown.min()

    # Number of signal days
    signal_days = (
        df["Position"] != 0
    ).sum()

    # --------------------------------------------------------
    # RESULT
    # --------------------------------------------------------

    result = {
        "Ticker": ticker,
        "Strategy Return %": round(
            cumulative_return * 100, 2
        ),
        "Buy & Hold %": round(
            buy_hold_return * 100, 2
        ),
        "Win Rate %": round(
            win_rate, 2
        ),
        "Max Drawdown %": round(
            max_drawdown * 100, 2
        ),
        "Signal Days": int(signal_days)
    }

    return result


# ============================================================
# RUN BACKTEST
# ============================================================

backtest_results = []

for ticker in tickers:

    result = backtest_ticker(ticker)

    if result is not None:
        backtest_results.append(result)


# ------------------------------------------------------------
# CREATE RESULTS TABLE
# ------------------------------------------------------------

backtest_df = pd.DataFrame(backtest_results)

if not backtest_df.empty:

    backtest_df = backtest_df.sort_values(
        by="Strategy Return %",
        ascending=False
    ).reset_index(drop=True)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print()
print("=" * 100)
print("Q-SCANNER — HISTORICAL BACKTEST RESULTS")
print("=" * 100)

display(backtest_df)

Backtesting AAPL...
Backtesting MSFT...
Backtesting NVDA...
Backtesting AMZN...
Backtesting META...
Backtesting GOOGL...
Backtesting TSLA...

Q-SCANNER — HISTORICAL BACKTEST RESULTS


,Ticker,Strategy Return %,Buy & Hold %,Win Rate %,Max Drawdown %,Signal Days
0,AAPL,32.63,121.08,51.10,-35.69,906
1,META,0.34,79.56,50.77,-51.63,906
2,GOOGL,-2.87,133.03,51.22,-53.76,904
3,NVDA,-22.03,672.00,47.90,-73.15,927
4,AMZN,-47.92,46.46,49.03,-77.26,924
5,MSFT,-56.40,55.21,48.89,-63.67,902
6,TSLA,-75.18,2.54,48.53,-85.74,954


In [13]:
# ============================================================
# Q-SCANNER V4 — SIGNAL ENGINE
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
from ta.trend import SMAIndicator
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands


def _flatten_yfinance_columns(df):
    """Safely convert yfinance MultiIndex columns to simple column names."""
    df = df.copy()

    if isinstance(df.columns, pd.MultiIndex):
        # For a single ticker, the first level is normally OHLCV.
        df.columns = [str(col[0]) for col in df.columns]

    # Remove any accidental duplicate columns.
    df = df.loc[:, ~df.columns.duplicated()]

    return df


def generate_v4_signal(df):
    """
    Calculate indicators and return a DataFrame containing one row per
    completed trading day.

    IMPORTANT: the return statement is inside this function.
    """

    if df is None or df.empty:
        return pd.DataFrame()

    df = _flatten_yfinance_columns(df)

    required = ["Close"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Make Close a clean numeric Series.
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

    # Technical indicators
    df["SMA_20"] = SMAIndicator(
        close=df["Close"], window=20
    ).sma_indicator()

    df["SMA_50"] = SMAIndicator(
        close=df["Close"], window=50
    ).sma_indicator()

    df["RSI"] = RSIIndicator(
        close=df["Close"], window=14
    ).rsi()

    bb = BollingerBands(
        close=df["Close"],
        window=20,
        window_dev=2
    )

    df["BB_Upper"] = bb.bollinger_hband()
    df["BB_Middle"] = bb.bollinger_mavg()
    df["BB_Lower"] = bb.bollinger_lband()

    # 10-day percentage momentum
    df["Momentum"] = df["Close"].pct_change(10) * 100

    # Keep only complete rows.
    df = df.dropna().copy()

    if df.empty:
        return pd.DataFrame()

    scores = []
    signals = []
    reasons_list = []

    for _, row in df.iterrows():

        price = float(row["Close"])
        sma20 = float(row["SMA_20"])
        sma50 = float(row["SMA_50"])
        rsi = float(row["RSI"])
        bb_upper = float(row["BB_Upper"])
        bb_lower = float(row["BB_Lower"])
        momentum = float(row["Momentum"])

        score = 0
        reasons = []

        # 1. Price vs SMA20
        if price > sma20:
            score += 1
            reasons.append("Price above SMA20")
        else:
            score -= 1
            reasons.append("Price below SMA20")

        # 2. SMA20 vs SMA50
        if sma20 > sma50:
            score += 1
            reasons.append("SMA20 above SMA50")
        else:
            score -= 1
            reasons.append("SMA20 below SMA50")

        # 3. RSI
        if 50 <= rsi < 65:
            score += 1
            reasons.append("RSI bullish")
        elif 65 <= rsi < 70:
            score += 0.5
            reasons.append("RSI strong")
        elif 30 < rsi < 50:
            score -= 1
            reasons.append("RSI bearish")
        elif rsi <= 30:
            score += 0.5
            reasons.append("RSI oversold / reversal")
        else:  # rsi >= 70
            score -= 1
            reasons.append("RSI overbought")

        # 4. Momentum
        if momentum > 0:
            score += 1
            reasons.append("Positive momentum")
        else:
            score -= 1
            reasons.append("Negative momentum")

        # 5. Bollinger Bands
        if price >= bb_upper:
            score -= 0.5
            reasons.append("Potentially overbought")
        elif price <= bb_lower:
            score += 0.5
            reasons.append("Potentially oversold")
        else:
            reasons.append("Price inside Bollinger Bands")

        # Final classification
        if score >= 3:
            signal = "BUY"
        elif score <= -3:
            signal = "SELL"
        else:
            signal = "NEUTRAL"

        scores.append(round(score, 2))
        signals.append(signal)
        reasons_list.append("; ".join(reasons))

    df["Score"] = scores
    df["Signal"] = signals
    df["Reasons"] = reasons_list

    return df


In [14]:
# ============================================================
# RUN Q-SCANNER V4 — CURRENT MARKET SIGNALS
# ============================================================

TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "TSLA"]

START_DATE = "2022-01-01"

results = []

for ticker in TICKERS:

    print(f"Scanning {ticker}...")

    try:
        data = yf.download(
            ticker,
            start=START_DATE,
            auto_adjust=True,
            progress=False
        )

        if data is None or data.empty:
            print(f"{ticker}: No data returned.")
            continue

        scanned = generate_v4_signal(data)

        # Protect against the old NoneType/.iloc problem.
        if scanned is None:
            print(f"{ticker}: Signal engine returned None.")
            continue

        if scanned.empty:
            print(f"{ticker}: Not enough valid data after indicators.")
            continue

        latest = scanned.iloc[-1].copy()
        latest["Ticker"] = ticker

        results.append(latest)

    except Exception as e:
        print(f"Error scanning {ticker}: {type(e).__name__}: {e}")


# ============================================================
# FINAL TABLE
# ============================================================

if results:

    scanner_results = pd.DataFrame(results)

    scanner_results["Signal Strength"] = (
        scanner_results["Score"].abs()
    )

    scanner_results = scanner_results.sort_values(
        by=["Signal Strength", "Score"],
        ascending=[False, False]
    ).reset_index(drop=True)

    print()
    print("=" * 120)
    print("Q-SCANNER V4 — CURRENT MARKET SIGNALS")
    print("=" * 120)

    display(
        scanner_results[
            [
                "Ticker",
                "Close",
                "SMA_20",
                "SMA_50",
                "RSI",
                "Momentum",
                "Score",
                "Signal",
                "Reasons"
            ]
        ]
    )

else:
    print("No assets were successfully scanned.")


Scanning AAPL...
Scanning MSFT...
Scanning NVDA...
Scanning AMZN...
Scanning META...
Scanning GOOGL...
Scanning TSLA...

Q-SCANNER V4 — CURRENT MARKET SIGNALS


,Ticker,Close,SMA_20,SMA_50,RSI,Momentum,Score,Signal,Reasons
0,MSFT,505.795013,495.644011,443.683984,63.400805,4.667458,4.0,BUY,Price above SMA20; SMA20 above SMA50; RSI bull...
1,NVDA,232.460007,220.187001,210.608400,61.615633,8.261925,4.0,BUY,Price above SMA20; SMA20 above SMA50; RSI bull...
2,GOOGL,341.339996,343.754999,348.626999,46.892931,-1.009225,-4.0,SELL,Price below SMA20; SMA20 below SMA50; RSI bear...
3,AAPL,326.179993,313.450499,315.111968,60.833245,5.440435,2.0,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bull...
4,META,608.820007,576.483496,595.332595,59.869069,10.714672,2.0,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bull...
5,AMZN,258.910004,262.216998,253.957399,49.906334,0.108262,0.0,NEUTRAL,Price below SMA20; SMA20 above SMA50; RSI bear...
6,TSLA,359.940002,348.732500,358.064999,53.145323,-0.804713,0.0,NEUTRAL,Price above SMA20; SMA20 below SMA50; RSI bull...


In [15]:
 # ============================================================
# Q-SCANNER V4 — PROPER HISTORICAL BACKTEST
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

BACKTEST_TICKERS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META",
    "GOOGL",
    "TSLA"
]

BACKTEST_PERIOD = "5y"

TRANSACTION_COST = 0.001   # 0.10% per position change
TRADING_DAYS = 252


# ------------------------------------------------------------
# BACKTEST FUNCTION
# ------------------------------------------------------------

def backtest_v4(ticker):

    print(f"Backtesting {ticker}...")

    df = yf.download(
        ticker,
        period=BACKTEST_PERIOD,
        interval="1d",
        auto_adjust=True,
        progress=False
    )

    if df.empty:
        return None

    # Handle yfinance MultiIndex
    df = _flatten_yfinance_columns(df)

    if "Close" not in df.columns:
        return None

    df["Close"] = pd.to_numeric(
        df["Close"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # USE THE SAME V4 INDICATORS
    # --------------------------------------------------------

    df["SMA_20"] = df["Close"].rolling(20).mean()
    df["SMA_50"] = df["Close"].rolling(50).mean()

    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["RSI"] = 100 - (100 / (1 + rs))

    df["BB_Middle"] = df["Close"].rolling(20).mean()
    df["BB_STD"] = df["Close"].rolling(20).std()

    df["BB_Upper"] = (
        df["BB_Middle"] +
        2 * df["BB_STD"]
    )

    df["BB_Lower"] = (
        df["BB_Middle"] -
        2 * df["BB_STD"]
    )

    df["Momentum"] = (
        df["Close"].pct_change(10) * 100
    )

    df = df.dropna().copy()

    if df.empty:
        return None


    # --------------------------------------------------------
    # RECREATE V4 SCORE
    # --------------------------------------------------------

    scores = []

    for _, row in df.iterrows():

        price = float(row["Close"])
        sma20 = float(row["SMA_20"])
        sma50 = float(row["SMA_50"])
        rsi = float(row["RSI"])
        bb_upper = float(row["BB_Upper"])
        bb_lower = float(row["BB_Lower"])
        momentum = float(row["Momentum"])

        score = 0

        # 1. Price vs SMA20
        if price > sma20:
            score += 1
        else:
            score -= 1

        # 2. SMA20 vs SMA50
        if sma20 > sma50:
            score += 1
        else:
            score -= 1

        # 3. RSI
        if 50 <= rsi < 65:
            score += 1

        elif 65 <= rsi < 70:
            score += 0.5

        elif 30 < rsi < 50:
            score -= 1

        elif rsi <= 30:
            score += 0.5

        else:
            score -= 1

        # 4. Momentum
        if momentum > 0:
            score += 1
        else:
            score -= 1

        # 5. Bollinger Bands
        if price >= bb_upper:
            score -= 0.5

        elif price <= bb_lower:
            score += 0.5

        scores.append(score)


    df["Score"] = scores


    # --------------------------------------------------------
    # V4 SIGNAL
    # --------------------------------------------------------

    df["Signal"] = "NEUTRAL"

    df.loc[
        df["Score"] >= 3,
        "Signal"
    ] = "BUY"

    df.loc[
        df["Score"] <= -3,
        "Signal"
    ] = "SELL"


    # --------------------------------------------------------
    # POSITION
    # --------------------------------------------------------

    df["Position"] = 0

    df.loc[
        df["Signal"] == "BUY",
        "Position"
    ] = 1

    df.loc[
        df["Signal"] == "SELL",
        "Position"
    ] = -1


    # --------------------------------------------------------
    # IMPORTANT:
    # SIGNAL IS GENERATED AT TODAY'S CLOSE.
    # THEREFORE IT IS USED FROM THE NEXT TRADING DAY.
    # --------------------------------------------------------

    df["Position_Used"] = df["Position"].shift(1)

    df["Market_Return"] = (
        df["Close"].pct_change()
    )

    df["Strategy_Return"] = (
        df["Position_Used"] *
        df["Market_Return"]
    )


    # --------------------------------------------------------
    # TRANSACTION COST
    # --------------------------------------------------------

    df["Position_Change"] = (
        df["Position_Used"]
        .diff()
        .abs()
        .fillna(0)
    )

    df["Strategy_Return"] = (
        df["Strategy_Return"]
        -
        df["Position_Change"] *
        TRANSACTION_COST
    )


    df = df.dropna().copy()


    # --------------------------------------------------------
    # EQUITY CURVES
    # --------------------------------------------------------

    df["Strategy_Equity"] = (
        1 + df["Strategy_Return"]
    ).cumprod()

    df["BuyHold_Equity"] = (
        1 + df["Market_Return"]
    ).cumprod()


    # --------------------------------------------------------
    # RETURNS
    # --------------------------------------------------------

    strategy_return = (
        df["Strategy_Equity"].iloc[-1] - 1
    )

    buy_hold_return = (
        df["BuyHold_Equity"].iloc[-1] - 1
    )


    # --------------------------------------------------------
    # CAGR
    # --------------------------------------------------------

    years = (
        (df.index[-1] - df.index[0]).days / 365.25
    )

    strategy_cagr = (
        df["Strategy_Equity"].iloc[-1]
        ** (1 / years)
        - 1
    )

    buy_hold_cagr = (
        df["BuyHold_Equity"].iloc[-1]
        ** (1 / years)
        - 1
    )


    # --------------------------------------------------------
    # MAX DRAWDOWN
    # --------------------------------------------------------

    running_max = (
        df["Strategy_Equity"].cummax()
    )

    drawdown = (
        df["Strategy_Equity"] /
        running_max
    ) - 1

    max_drawdown = drawdown.min()


    # --------------------------------------------------------
    # SHARPE RATIO
    # --------------------------------------------------------

    daily_std = df["Strategy_Return"].std()

    if daily_std != 0:

        sharpe = (
            df["Strategy_Return"].mean()
            /
            daily_std
        ) * np.sqrt(TRADING_DAYS)

    else:
        sharpe = 0


    # --------------------------------------------------------
    # TRADES
    # --------------------------------------------------------

    trades = (
        df["Position_Used"]
        .diff()
        .abs()
        .fillna(0)
        .gt(0)
        .sum()
    )


    # --------------------------------------------------------
    # WIN RATE
    # --------------------------------------------------------

    active_returns = df.loc[
        df["Position_Used"] != 0,
        "Strategy_Return"
    ]

    if len(active_returns) > 0:

        win_rate = (
            active_returns > 0
        ).mean()

    else:

        win_rate = 0


    # --------------------------------------------------------
    # PROFIT FACTOR
    # --------------------------------------------------------

    winning_returns = active_returns[
        active_returns > 0
    ].sum()

    losing_returns = abs(
        active_returns[
            active_returns < 0
        ].sum()
    )

    if losing_returns > 0:

        profit_factor = (
            winning_returns /
            losing_returns
        )

    else:

        profit_factor = np.inf


    # --------------------------------------------------------
    # EXPOSURE
    # --------------------------------------------------------

    exposure = (
        df["Position_Used"] != 0
    ).mean()


    # --------------------------------------------------------
    # RESULT
    # --------------------------------------------------------

    return {
        "Ticker": ticker,

        "Strategy Return %":
            round(strategy_return * 100, 2),

        "Buy & Hold %":
            round(buy_hold_return * 100, 2),

        "Strategy CAGR %":
            round(strategy_cagr * 100, 2),

        "Buy & Hold CAGR %":
            round(buy_hold_cagr * 100, 2),

        "Max Drawdown %":
            round(max_drawdown * 100, 2),

        "Sharpe":
            round(sharpe, 2),

        "Win Rate %":
            round(win_rate * 100, 2),

        "Profit Factor":
            round(profit_factor, 2)
            if np.isfinite(profit_factor)
            else np.inf,

        "Trades":
            int(trades),

        "Exposure %":
            round(exposure * 100, 2)
    }


# ============================================================
# RUN BACKTEST
# ============================================================

results = []

for ticker in BACKTEST_TICKERS:

    try:

        result = backtest_v4(ticker)

        if result is not None:
            results.append(result)

    except Exception as e:

        print(
            f"{ticker}: ERROR -> "
            f"{type(e).__name__}: {e}"
        )


# ============================================================
# RESULTS TABLE
# ============================================================

backtest_v4_df = pd.DataFrame(results)

if not backtest_v4_df.empty:

    backtest_v4_df = (
        backtest_v4_df
        .sort_values(
            by="Strategy CAGR %",
            ascending=False
        )
        .reset_index(drop=True)
    )

    print()
    print("=" * 120)
    print("Q-SCANNER V4 — PROPER HISTORICAL BACKTEST")
    print("=" * 120)

    display(backtest_v4_df)

else:

    print("No backtest results were generated.")

Backtesting AAPL...
Backtesting MSFT...
Backtesting NVDA...
Backtesting AMZN...
Backtesting META...
Backtesting GOOGL...
Backtesting TSLA...

Q-SCANNER V4 — PROPER HISTORICAL BACKTEST


,Ticker,Strategy Return %,Buy & Hold %,Strategy CAGR %,Buy & Hold CAGR %,Max Drawdown %,Sharpe,Win Rate %,Profit Factor,Trades,Exposure %
0,AAPL,51.38,122.61,9.02,18.14,-13.71,0.63,52.74,1.28,207,31.78
1,GOOGL,21.26,132.18,4.10,19.19,-38.91,0.30,51.55,1.15,277,32.03
2,NVDA,6.19,677.23,1.26,53.30,-38.80,0.18,47.06,1.08,239,33.86
3,AMZN,-6.53,45.95,-1.40,8.20,-47.12,0.05,50.12,1.06,253,34.44
4,META,-20.18,76.70,-4.59,12.59,-41.30,-0.08,50.00,1.01,226,29.21
5,MSFT,-35.67,56.81,-8.78,9.83,-46.93,-0.53,48.48,0.90,261,30.12
6,TSLA,-42.11,6.66,-10.77,1.35,-64.69,-0.20,45.99,0.96,224,26.89


In [16]:
# ============================================================
# Q-SCANNER V5 — ROBUST PERFORMANCE & PORTFOLIO DIAGNOSTICS
# ============================================================
#
# PURPOSE:
#   Evaluate the EXISTING V4 strategy properly.
#
# IMPORTANT:
#   The trading rules are NOT changed here.
#   V5 only improves how we measure the strategy.
#
# Metrics:
#   - Total Strategy Return
#   - Buy & Hold Return
#   - CAGR
#   - Annualised Volatility
#   - Sharpe Ratio
#   - Sortino Ratio
#   - Maximum Drawdown
#   - Calmar Ratio
#   - Win Rate
#   - Profit Factor
#   - Expectancy
#   - Average Win
#   - Average Loss
#   - Number of Trades
#   - Exposure
#   - Year-by-year performance
#   - Benchmark comparison
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

TICKERS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META",
    "GOOGL",
    "TSLA"
]

BENCHMARK = "SPY"

START_DATE = "2021-01-01"

TRANSACTION_COST = 0.001      # 0.10%
RISK_FREE_RATE = 0.0          # Simplified for now


# ============================================================
# 1. INDICATOR CALCULATION
# ============================================================

def calculate_v5_indicators(df):

    df = df.copy()

    # Handle yfinance MultiIndex
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.loc[:, ~df.columns.duplicated()]

    df["Close"] = pd.to_numeric(
        df["Close"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # SMA
    # --------------------------------------------------------

    df["SMA_20"] = (
        df["Close"]
        .rolling(20)
        .mean()
    )

    df["SMA_50"] = (
        df["Close"]
        .rolling(50)
        .mean()
    )

    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    delta = df["Close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["RSI"] = (
        100 - (100 / (1 + rs))
    )

    # --------------------------------------------------------
    # Bollinger Bands
    # --------------------------------------------------------

    df["BB_Middle"] = (
        df["Close"]
        .rolling(20)
        .mean()
    )

    df["BB_STD"] = (
        df["Close"]
        .rolling(20)
        .std()
    )

    df["BB_Upper"] = (
        df["BB_Middle"]
        + 2 * df["BB_STD"]
    )

    df["BB_Lower"] = (
        df["BB_Middle"]
        - 2 * df["BB_STD"]
    )

    # --------------------------------------------------------
    # 10-DAY MOMENTUM
    # --------------------------------------------------------

    df["Momentum"] = (
        df["Close"].pct_change(10)
    )

    return df.dropna().copy()


# ============================================================
# 2. V4 SCORING RULES
# ============================================================

def calculate_v5_score(row):

    score = 0

    price = row["Close"]
    sma20 = row["SMA_20"]
    sma50 = row["SMA_50"]
    rsi = row["RSI"]
    momentum = row["Momentum"]

    # --------------------------------------------------------
    # PRICE VS SMA20
    # --------------------------------------------------------

    if price > sma20:
        score += 1
    else:
        score -= 1

    # --------------------------------------------------------
    # SMA20 VS SMA50
    # --------------------------------------------------------

    if sma20 > sma50:
        score += 1
    else:
        score -= 1

    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    if 50 <= rsi < 65:

        score += 1

    elif 65 <= rsi < 70:

        score += 0.5

    elif 30 < rsi < 50:

        score -= 1

    elif rsi <= 30:

        score += 0.5

    elif rsi >= 70:

        score -= 1

    # --------------------------------------------------------
    # MOMENTUM
    # --------------------------------------------------------

    if momentum > 0:
        score += 1
    else:
        score -= 1

    # --------------------------------------------------------
    # BOLLINGER
    # --------------------------------------------------------

    if price >= row["BB_Upper"]:

        score -= 0.5

    elif price <= row["BB_Lower"]:

        score += 0.5

    return score


# ============================================================
# 3. PERFORMANCE METRICS
# ============================================================

def calculate_metrics(df):

    returns = df["Strategy_Return"].dropna()

    if len(returns) == 0:
        return {}

    # --------------------------------------------------------
    # TOTAL RETURN
    # --------------------------------------------------------

    total_return = (
        (1 + returns).prod() - 1
    )

    # --------------------------------------------------------
    # BUY & HOLD
    # --------------------------------------------------------

    buy_hold_return = (
        (1 + df["Market_Return"].dropna()).prod() - 1
    )

    # --------------------------------------------------------
    # YEARS
    # --------------------------------------------------------

    days = (
        df.index[-1] - df.index[0]
    ).days

    years = days / 365.25

    if years > 0:

        cagr = (
            (1 + total_return) ** (1 / years)
            - 1
        )

        buy_hold_cagr = (
            (1 + buy_hold_return) ** (1 / years)
            - 1
        )

    else:

        cagr = np.nan
        buy_hold_cagr = np.nan

    # --------------------------------------------------------
    # VOLATILITY
    # --------------------------------------------------------

    annual_volatility = (
        returns.std() * np.sqrt(252)
    )

    # --------------------------------------------------------
    # SHARPE
    # --------------------------------------------------------

    if returns.std() != 0:

        sharpe = (
            (returns.mean() - RISK_FREE_RATE / 252)
            / returns.std()
        ) * np.sqrt(252)

    else:

        sharpe = np.nan

    # --------------------------------------------------------
    # SORTINO
    # --------------------------------------------------------

    downside_returns = returns[
        returns < 0
    ]

    downside_deviation = (
        downside_returns.std()
        * np.sqrt(252)
    )

    if downside_deviation != 0:

        sortino = (
            (returns.mean() * 252)
            / downside_deviation
        )

    else:

        sortino = np.nan

    # --------------------------------------------------------
    # EQUITY CURVE
    # --------------------------------------------------------

    equity = (
        1 + returns
    ).cumprod()

    running_max = equity.cummax()

    drawdown = (
        equity / running_max
    ) - 1

    max_drawdown = drawdown.min()

    # --------------------------------------------------------
    # CALMAR
    # --------------------------------------------------------

    if max_drawdown != 0:

        calmar = (
            cagr / abs(max_drawdown)
        )

    else:

        calmar = np.nan

    # --------------------------------------------------------
    # ACTIVE DAYS
    # --------------------------------------------------------

    active = df[
        df["Position"] != 0
    ]

    active_returns = (
        active["Strategy_Return"]
    )

    if len(active_returns) > 0:

        win_rate = (
            active_returns > 0
        ).mean()

        average_win = (
            active_returns[
                active_returns > 0
            ].mean()
        )

        average_loss = (
            active_returns[
                active_returns < 0
            ].mean()
        )

        gross_profit = (
            active_returns[
                active_returns > 0
            ].sum()
        )

        gross_loss = abs(
            active_returns[
                active_returns < 0
            ].sum()
        )

        if gross_loss != 0:
            profit_factor = (
                gross_profit / gross_loss
            )
        else:
            profit_factor = np.inf

        expectancy = (
            active_returns.mean()
        )

    else:

        win_rate = 0
        average_win = np.nan
        average_loss = np.nan
        profit_factor = np.nan
        expectancy = np.nan

    # --------------------------------------------------------
    # TRADES
    # --------------------------------------------------------

    position_change = (
        df["Position"].diff().fillna(
            df["Position"]
        )
    )

    entries = (
        (df["Position"] != 0)
        & (df["Position"].shift(1).fillna(0) == 0)
    ).sum()

    exits = (
        (df["Position"] == 0)
        & (df["Position"].shift(1).fillna(0) != 0)
    ).sum()

    flips = (
        (
            (df["Position"] != 0)
            & (df["Position"].shift(1).fillna(0) != 0)
            & (
                df["Position"]
                != df["Position"].shift(1)
            )
        )
    ).sum()

    # A flip represents closing one position
    # and opening another.
    trade_count = int(
        entries + flips
    )

    # --------------------------------------------------------
    # EXPOSURE
    # --------------------------------------------------------

    exposure = (
        df["Position"] != 0
    ).mean()

    # --------------------------------------------------------
    # RETURN / DRAWDOWN RATIO
    # --------------------------------------------------------

    return {
        "Strategy Return %": total_return * 100,
        "Buy & Hold %": buy_hold_return * 100,
        "CAGR %": cagr * 100,
        "Buy & Hold CAGR %": buy_hold_cagr * 100,
        "Annual Volatility %": annual_volatility * 100,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max Drawdown %": max_drawdown * 100,
        "Calmar": calmar,
        "Win Rate %": win_rate * 100,
        "Profit Factor": profit_factor,
        "Expectancy %": expectancy * 100,
        "Average Win %": average_win * 100,
        "Average Loss %": average_loss * 100,
        "Trades": trade_count,
        "Exposure %": exposure * 100
    }


# ============================================================
# 4. BACKTEST ONE ASSET
# ============================================================

def backtest_v5(ticker):

    print(f"Backtesting {ticker}...")

    df = yf.download(
        ticker,
        start=START_DATE,
        auto_adjust=True,
        progress=False
    )

    if df is None or df.empty:
        return None, None

    df = calculate_v5_indicators(df)

    if df.empty:
        return None, None

    # --------------------------------------------------------
    # SCORE
    # --------------------------------------------------------

    df["Score"] = df.apply(
        calculate_v5_score,
        axis=1
    )

    # --------------------------------------------------------
    # SIGNAL
    # --------------------------------------------------------

    df["Signal"] = "NEUTRAL"

    df.loc[
        df["Score"] >= 3,
        "Signal"
    ] = "BUY"

    df.loc[
        df["Score"] <= -3,
        "Signal"
    ] = "SELL"

    # --------------------------------------------------------
    # POSITION
    # --------------------------------------------------------

    df["Position"] = 0

    df.loc[
        df["Score"] >= 3,
        "Position"
    ] = 1

    df.loc[
        df["Score"] <= -3,
        "Position"
    ] = -1

    # --------------------------------------------------------
    # NEXT-DAY MARKET RETURN
    # --------------------------------------------------------

    df["Market_Return"] = (
        df["Close"].pct_change().shift(-1)
    )

    # --------------------------------------------------------
    # STRATEGY RETURN
    # --------------------------------------------------------

    df["Strategy_Return"] = (
        df["Position"]
        * df["Market_Return"]
    )

    # --------------------------------------------------------
    # TRANSACTION COST
    # --------------------------------------------------------

    df["Position_Change"] = (
        df["Position"]
        .diff()
        .abs()
    )

    df["Strategy_Return"] = (
        df["Strategy_Return"]
        - (
            df["Position_Change"]
            * TRANSACTION_COST
        )
    )

    # Remove rows without a next-day return
    df = df.dropna(
        subset=["Strategy_Return"]
    ).copy()

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    metrics = calculate_metrics(df)

    return metrics, df


# ============================================================
# 5. RUN ALL ASSETS
# ============================================================

all_metrics = []
all_data = {}

for ticker in TICKERS:

    try:

        metrics, df = backtest_v5(ticker)

        if metrics:

            metrics["Ticker"] = ticker

            all_metrics.append(metrics)

            all_data[ticker] = df

    except Exception as e:

        print(
            f"{ticker}: ERROR -> "
            f"{type(e).__name__}: {e}"
        )


# ============================================================
# 6. RESULTS TABLE
# ============================================================

v5_results = pd.DataFrame(
    all_metrics
)

if not v5_results.empty:

    column_order = [
        "Ticker",
        "Strategy Return %",
        "Buy & Hold %",
        "CAGR %",
        "Buy & Hold CAGR %",
        "Annual Volatility %",
        "Sharpe",
        "Sortino",
        "Max Drawdown %",
        "Calmar",
        "Win Rate %",
        "Profit Factor",
        "Expectancy %",
        "Average Win %",
        "Average Loss %",
        "Trades",
        "Exposure %"
    ]

    v5_results = (
        v5_results[column_order]
        .sort_values(
            by="Sharpe",
            ascending=False
        )
        .reset_index(drop=True)
    )

    # Round for display
    numeric_columns = (
        v5_results.columns[
            v5_results.columns != "Ticker"
        ]
    )

    v5_results[numeric_columns] = (
        v5_results[numeric_columns]
        .round(2)
    )


# ============================================================
# DISPLAY
# ============================================================

print()
print("=" * 140)
print("Q-SCANNER V5 — ROBUST PERFORMANCE ANALYSIS")
print("=" * 140)

display(v5_results)


# ============================================================
# 7. YEAR-BY-YEAR PERFORMANCE
# ============================================================

yearly_results = []

for ticker, df in all_data.items():

    temp = df.copy()

    temp["Year"] = temp.index.year

    yearly = (
        temp
        .groupby("Year")
        .agg(
            Strategy_Return=(
                "Strategy_Return",
                lambda x: (1 + x).prod() - 1
            ),
            Buy_Hold_Return=(
                "Market_Return",
                lambda x: (1 + x).prod() - 1
            )
        )
        .reset_index()
    )

    yearly["Ticker"] = ticker

    yearly_results.append(
        yearly
    )


yearly_df = pd.concat(
    yearly_results,
    ignore_index=True
)

yearly_df["Strategy Return %"] = (
    yearly_df["Strategy_Return"] * 100
)

yearly_df["Buy & Hold %"] = (
    yearly_df["Buy_Hold_Return"] * 100
)

yearly_df = yearly_df[
    [
        "Year",
        "Ticker",
        "Strategy Return %",
        "Buy & Hold %"
    ]
]

yearly_df[
    ["Strategy Return %", "Buy & Hold %"]
] = yearly_df[
    ["Strategy Return %", "Buy & Hold %"]
].round(2)

yearly_df = yearly_df.sort_values(
    ["Year", "Ticker"]
).reset_index(drop=True)


print()
print("=" * 100)
print("Q-SCANNER V5 — YEAR-BY-YEAR RESULTS")
print("=" * 100)

display(yearly_df)


# ============================================================
# 8. BENCHMARK — SPY
# ============================================================

print()
print("Downloading benchmark...")

spy = yf.download(
    BENCHMARK,
    start=START_DATE,
    auto_adjust=True,
    progress=False
)

if not spy.empty:

    if isinstance(spy.columns, pd.MultiIndex):
        spy.columns = spy.columns.get_level_values(0)

    spy_close = pd.to_numeric(
        spy["Close"],
        errors="coerce"
    ).dropna()

    spy_return = (
        spy_close.iloc[-1]
        / spy_close.iloc[0]
    ) - 1

    spy_years = (
        spy_close.index[-1]
        - spy_close.index[0]
    ).days / 365.25

    spy_cagr = (
        (1 + spy_return)
        ** (1 / spy_years)
        - 1
    )

    print()
    print("=" * 80)
    print("BENCHMARK — SPY")
    print("=" * 80)
    print(
        f"SPY Total Return: "
        f"{spy_return * 100:.2f}%"
    )
    print(
        f"SPY CAGR: "
        f"{spy_cagr * 100:.2f}%"
    )


# ============================================================
# 9. SIMPLE INTERPRETATION
# ============================================================

print()
print("=" * 100)
print("Q-SCANNER V5 — AUTOMATED DIAGNOSTIC")
print("=" * 100)

for _, row in v5_results.iterrows():

    ticker = row["Ticker"]
    sharpe = row["Sharpe"]
    profit_factor = row["Profit Factor"]
    cagr = row["CAGR %"]
    drawdown = row["Max Drawdown %"]
    strategy_return = row["Strategy Return %"]
    buy_hold = row["Buy & Hold %"]

    print()
    print(f"{ticker}")
    print("-" * 40)

    print(
        f"Strategy Return: {strategy_return:.2f}%"
    )

    print(
        f"Buy & Hold:      {buy_hold:.2f}%"
    )

    print(
        f"CAGR:            {cagr:.2f}%"
    )

    print(
        f"Sharpe:          {sharpe:.2f}"
    )

    print(
        f"Profit Factor:   {profit_factor:.2f}"
    )

    print(
        f"Max Drawdown:    {drawdown:.2f}%"
    )

    # Basic diagnostic
    if (
        sharpe >= 1
        and profit_factor > 1.2
        and cagr > 0
    ):
        verdict = "PROMISING"

    elif (
        sharpe > 0
        and profit_factor > 1
        and cagr > 0
    ):
        verdict = "WEAK POSITIVE EDGE"

    elif (
        sharpe <= 0
        or profit_factor < 1
        or cagr < 0
    ):
        verdict = "NOT ROBUST"

    else:
        verdict = "NEEDS INVESTIGATION"

    print(
        f"Diagnostic:      {verdict}"
    )

Backtesting AAPL...
Backtesting MSFT...
Backtesting NVDA...
Backtesting AMZN...
Backtesting META...
Backtesting GOOGL...
Backtesting TSLA...

Q-SCANNER V5 — ROBUST PERFORMANCE ANALYSIS


,Ticker,Strategy Return %,Buy & Hold %,CAGR %,Buy & Hold CAGR %,Annual Volatility %,Sharpe,Sortino,Max Drawdown %,Calmar,Win Rate %,Profit Factor,Expectancy %,Average Win %,Average Loss %,Trades,Exposure %
0,GOOGL,47.09,230.98,7.32,24.49,18.78,0.47,0.44,-39.76,0.18,53.26,1.21,0.14,1.54,-1.44,155,32.39
1,AAPL,34.16,168.55,5.53,19.81,15.03,0.43,0.40,-13.71,0.40,51.98,1.20,0.11,1.25,-1.13,119,31.22
2,NVDA,24.41,1654.03,4.08,68.91,27.63,0.28,0.30,-38.80,0.11,47.82,1.12,0.13,2.55,-2.10,131,31.66
3,AMZN,-14.78,64.85,-2.88,9.58,21.74,-0.03,-0.02,-47.12,-0.06,49.55,1.03,0.02,1.65,-1.58,138,32.31
4,META,-20.44,116.31,-4.10,15.16,23.06,-0.07,-0.06,-41.30,-0.10,49.65,1.01,0.01,1.79,-1.75,133,31.22
5,TSLA,-44.88,54.17,-10.33,8.24,30.01,-0.21,-0.17,-64.69,-0.16,46.50,0.95,-0.06,2.81,-2.56,124,25.98
6,MSFT,-34.91,123.40,-7.56,15.85,14.49,-0.47,-0.39,-48.33,-0.16,48.86,0.91,-0.06,1.23,-1.29,146,28.75



Q-SCANNER V5 — YEAR-BY-YEAR RESULTS


,Year,Ticker,Strategy Return %,Buy & Hold %
0,2021,AAPL,-0.79,46.57
1,2021,AMZN,-10.64,8.69
2,2021,GOOGL,16.89,39.27
3,2021,META,0.22,19.20
4,2021,MSFT,-3.29,42.08
5,2021,NVDA,3.50,125.89
6,2021,TSLA,-20.93,70.96
7,2022,AAPL,4.01,-30.89
8,2022,AMZN,71.73,-49.64
9,2022,GOOGL,-20.05,-38.53




BENCHMARK — SPY
SPY Total Return: 125.56%
SPY CAGR: 15.44%

Q-SCANNER V5 — AUTOMATED DIAGNOSTIC

GOOGL
----------------------------------------
Strategy Return: 47.09%
Buy & Hold:      230.98%
CAGR:            7.32%
Sharpe:          0.47
Profit Factor:   1.21
Max Drawdown:    -39.76%
Diagnostic:      WEAK POSITIVE EDGE

AAPL
----------------------------------------
Strategy Return: 34.16%
Buy & Hold:      168.55%
CAGR:            5.53%
Sharpe:          0.43
Profit Factor:   1.20
Max Drawdown:    -13.71%
Diagnostic:      WEAK POSITIVE EDGE

NVDA
----------------------------------------
Strategy Return: 24.41%
Buy & Hold:      1654.03%
CAGR:            4.08%
Sharpe:          0.28
Profit Factor:   1.12
Max Drawdown:    -38.80%
Diagnostic:      WEAK POSITIVE EDGE

AMZN
----------------------------------------
Strategy Return: -14.78%
Buy & Hold:      64.85%
CAGR:            -2.88%
Sharpe:          -0.03
Profit Factor:   1.03
Max Drawdown:    -47.12%
Diagnostic:      NOT ROBUST

META
----

In [17]:
# ============================================================
# Q-SCANNER V6 — DEPENDENCY CHECK
# ============================================================

from ta.trend import ADXIndicator
from ta.volatility import AverageTrueRange

print("ADXIndicator: OK")
print("AverageTrueRange: OK")
print("Q-SCANNER V6 READY")

ADXIndicator: OK
AverageTrueRange: OK
Q-SCANNER V6 READY


In [18]:
# ============================================================
# Q-SCANNER V6 — ADVANCED INDICATORS
# ============================================================

def calculate_v6_indicators(df):

    df = df.copy()

    # --------------------------------------------------------
    # CLEAN DATA
    # --------------------------------------------------------

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [str(col[0]) for col in df.columns]

    df = df.loc[:, ~df.columns.duplicated()]

    for column in ["Open", "High", "Low", "Close", "Volume"]:
        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )

    # --------------------------------------------------------
    # MOVING AVERAGES
    # --------------------------------------------------------

    df["SMA_20"] = df["Close"].rolling(20).mean()
    df["SMA_50"] = df["Close"].rolling(50).mean()
    df["SMA_200"] = df["Close"].rolling(200).mean()

    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    df["RSI"] = RSIIndicator(
        close=df["Close"],
        window=14
    ).rsi()

    # --------------------------------------------------------
    # BOLLINGER BANDS
    # --------------------------------------------------------

    bb = BollingerBands(
        close=df["Close"],
        window=20,
        window_dev=2
    )

    df["BB_Upper"] = bb.bollinger_hband()
    df["BB_Middle"] = bb.bollinger_mavg()
    df["BB_Lower"] = bb.bollinger_lband()

    # --------------------------------------------------------
    # MOMENTUM
    # --------------------------------------------------------

    df["Momentum_10"] = (
        df["Close"].pct_change(10) * 100
    )

    # --------------------------------------------------------
    # ATR
    # --------------------------------------------------------

    atr = AverageTrueRange(
        high=df["High"],
        low=df["Low"],
        close=df["Close"],
        window=14
    )

    df["ATR"] = atr.average_true_range()

    # ATR as percentage of price
    df["ATR_Pct"] = (
        df["ATR"] / df["Close"] * 100
    )

    # --------------------------------------------------------
    # ADX
    # --------------------------------------------------------

    adx = ADXIndicator(
        high=df["High"],
        low=df["Low"],
        close=df["Close"],
        window=14
    )

    df["ADX"] = adx.adx()

    # --------------------------------------------------------
    # CLEAN
    # --------------------------------------------------------

    df = df.dropna().copy()

    return df

In [19]:
# ============================================================
# Q-SCANNER V6 — SIGNAL ENGINE
# ============================================================

def generate_v6_signal(row):

    score = 0
    reasons = []

    price = float(row["Close"])
    sma20 = float(row["SMA_20"])
    sma50 = float(row["SMA_50"])
    sma200 = float(row["SMA_200"])
    rsi = float(row["RSI"])
    momentum = float(row["Momentum_10"])
    adx = float(row["ADX"])

    # ========================================================
    # 1. LONG-TERM REGIME
    # ========================================================

    if price > sma200:
        score += 2
        reasons.append("Above SMA200")
    else:
        score -= 2
        reasons.append("Below SMA200")

    # ========================================================
    # 2. MEDIUM-TERM TREND
    # ========================================================

    if sma20 > sma50:
        score += 1
        reasons.append("SMA20 above SMA50")
    else:
        score -= 1
        reasons.append("SMA20 below SMA50")

    # ========================================================
    # 3. PRICE VS SMA20
    # ========================================================

    if price > sma20:
        score += 1
        reasons.append("Price above SMA20")
    else:
        score -= 1
        reasons.append("Price below SMA20")

    # ========================================================
    # 4. RSI
    # ========================================================

    if 50 <= rsi < 65:
        score += 1
        reasons.append("Healthy RSI")

    elif 65 <= rsi < 70:
        score += 0.5
        reasons.append("Strong RSI")

    elif rsi >= 70:
        score -= 1
        reasons.append("Overbought RSI")

    elif 30 <= rsi < 50:
        score -= 1
        reasons.append("Weak RSI")

    elif rsi < 30:
        score += 0.5
        reasons.append("Oversold RSI")

    # ========================================================
    # 5. MOMENTUM
    # ========================================================

    if momentum > 0:
        score += 1
        reasons.append("Positive momentum")
    else:
        score -= 1
        reasons.append("Negative momentum")

    # ========================================================
    # 6. ADX TREND STRENGTH
    # ========================================================

    if adx >= 25:
        score += 1
        reasons.append("Strong trend")

    elif adx < 15:
        score -= 0.5
        reasons.append("Weak trend")

    else:
        reasons.append("Moderate trend")

    # ========================================================
    # FINAL SIGNAL
    # ========================================================

    if score >= 5:
        signal = "BUY"

    elif score <= -4:
        signal = "SELL"

    else:
        signal = "NEUTRAL"

    return pd.Series({
        "V6 Score": round(score, 2),
        "V6 Signal": signal,
        "V6 Reasons": "; ".join(reasons)
    })

In [20]:
# ============================================================
# Q-SCANNER V6 — CURRENT MARKET SCANNER
# ============================================================

TICKERS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META",
    "GOOGL",
    "TSLA"
]

START_DATE = "2022-01-01"

v6_results = []

for ticker in TICKERS:

    print(f"Scanning {ticker}...")

    try:

        data = yf.download(
            ticker,
            start=START_DATE,
            auto_adjust=True,
            progress=False
        )

        if data.empty:
            continue

        data = calculate_v6_indicators(data)

        signal = data.apply(
            generate_v6_signal,
            axis=1
        )

        data = pd.concat(
            [data, signal],
            axis=1
        )

        latest = data.iloc[-1].copy()

        latest["Ticker"] = ticker

        v6_results.append(latest)

    except Exception as e:

        print(
            f"{ticker}: ERROR -> "
            f"{type(e).__name__}: {e}"
        )


# ============================================================
# RESULTS
# ============================================================

v6_current = pd.DataFrame(v6_results)

if not v6_current.empty:

    v6_current["Signal Strength"] = (
        v6_current["V6 Score"].abs()
    )

    v6_current = v6_current.sort_values(
        by=["Signal Strength", "V6 Score"],
        ascending=[False, False]
    ).reset_index(drop=True)

    print()
    print("=" * 130)
    print("Q-SCANNER V6 — CURRENT MARKET REGIME")
    print("=" * 130)

    display(
        v6_current[
            [
                "Ticker",
                "Close",
                "SMA_20",
                "SMA_50",
                "SMA_200",
                "RSI",
                "Momentum_10",
                "ADX",
                "ATR_Pct",
                "V6 Score",
                "V6 Signal",
                "V6 Reasons"
            ]
        ]
    )

Scanning AAPL...
Scanning MSFT...
Scanning NVDA...
Scanning AMZN...
Scanning META...
Scanning GOOGL...
Scanning TSLA...

Q-SCANNER V6 — CURRENT MARKET REGIME


,Ticker,Close,SMA_20,SMA_50,SMA_200,RSI,Momentum_10,ADX,ATR_Pct,V6 Score,V6 Signal,V6 Reasons
0,MSFT,505.690002,495.638760,443.681884,429.402864,63.331173,4.645727,38.088648,2.254769,7.0,BUY,Above SMA200; SMA20 above SMA50; Price above S...
1,NVDA,232.639999,220.196001,210.612000,196.536820,61.716925,8.345752,17.418413,3.124046,6.0,BUY,Above SMA200; SMA20 above SMA50; Price above S...
2,AAPL,325.769989,313.429999,315.103768,283.468025,60.325355,5.307898,16.050068,2.179223,4.0,NEUTRAL,Above SMA200; SMA20 below SMA50; Price above S...
3,GOOGL,341.500000,343.762999,348.630199,335.726051,47.013580,-0.962823,7.278219,2.297668,-2.5,NEUTRAL,Above SMA200; SMA20 below SMA50; Price below S...
4,TSLA,362.184998,348.844749,358.109899,399.598975,54.047429,-0.186019,23.468300,4.072795,-2.0,NEUTRAL,Below SMA200; SMA20 below SMA50; Price above S...
5,META,608.175476,576.451270,595.319705,621.563704,59.587568,10.597463,12.553663,3.262947,-0.5,NEUTRAL,Below SMA200; SMA20 below SMA50; Price above S...
6,AMZN,258.549988,262.198997,253.950199,239.123050,49.584557,-0.030939,16.529762,2.502786,0.0,NEUTRAL,Above SMA200; SMA20 above SMA50; Price below S...


In [21]:
# ============================================================
# Q-SCANNER V6 — WALK-FORWARD BACKTEST ENGINE
# ============================================================

import numpy as np
import pandas as pd
import yfinance as yf

TRANSACTION_COST = 0.001   # 0.10%

TICKERS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META",
    "GOOGL",
    "TSLA"
]


# ============================================================
# BACKTEST SIGNAL
# ============================================================

def calculate_v6_score(row):

    score = 0

    price = float(row["Close"])
    sma20 = float(row["SMA_20"])
    sma50 = float(row["SMA_50"])
    sma200 = float(row["SMA_200"])
    rsi = float(row["RSI"])
    momentum = float(row["Momentum_10"])
    adx = float(row["ADX"])

    # --------------------------------------------------------
    # 1. LONG-TERM REGIME
    # --------------------------------------------------------

    if price > sma200:
        score += 2
    else:
        score -= 2

    # --------------------------------------------------------
    # 2. MEDIUM-TERM TREND
    # --------------------------------------------------------

    if sma20 > sma50:
        score += 1
    else:
        score -= 1

    # --------------------------------------------------------
    # 3. SHORT-TERM TREND
    # --------------------------------------------------------

    if price > sma20:
        score += 1
    else:
        score -= 1

    # --------------------------------------------------------
    # 4. RSI
    # --------------------------------------------------------

    if 50 <= rsi < 65:
        score += 1

    elif 65 <= rsi < 70:
        score += 0.5

    elif rsi >= 70:
        score -= 1

    elif 30 <= rsi < 50:
        score -= 1

    elif rsi < 30:
        score += 0.5

    # --------------------------------------------------------
    # 5. MOMENTUM
    # --------------------------------------------------------

    if momentum > 0:
        score += 1
    else:
        score -= 1

    # --------------------------------------------------------
    # 6. ADX
    # --------------------------------------------------------

    if adx >= 25:
        score += 1

    elif adx < 15:
        score -= 0.5

    # --------------------------------------------------------
    # SIGNAL
    # --------------------------------------------------------

    if score >= 5:
        signal = "BUY"

    elif score <= -4:
        signal = "SELL"

    else:
        signal = "NEUTRAL"

    return score, signal


# ============================================================
# PREPARE DATA
# ============================================================

def prepare_v6_data(ticker):

    df = yf.download(
        ticker,
        start="2020-01-01",
        end="2026-08-31",
        auto_adjust=True,
        progress=False
    )

    if df.empty:
        return None

    df = calculate_v6_indicators(df)

    scores = []
    signals = []

    for _, row in df.iterrows():

        score, signal = calculate_v6_score(row)

        scores.append(score)
        signals.append(signal)

    df["Score"] = scores
    df["Signal"] = signals

    return df

In [22]:
# ============================================================
# Q-SCANNER V6 — STRATEGY SIMULATION
# ============================================================

def run_v6_strategy(df):

    df = df.copy()

    # --------------------------------------------------------
    # POSITION
    # --------------------------------------------------------

    # Long-only initially.
    #
    # BUY     = +1
    # NEUTRAL = 0
    # SELL    = 0
    #
    # We deliberately DON'T short yet.

    df["Position"] = 0

    df.loc[
        df["Signal"] == "BUY",
        "Position"
    ] = 1

    # --------------------------------------------------------
    # NEXT-DAY RETURN
    # --------------------------------------------------------

    df["Market_Return"] = (
        df["Close"].pct_change().shift(-1)
    )

    # Signal today -> position next trading day

    df["Strategy_Return"] = (
        df["Position"] *
        df["Market_Return"]
    )

    # --------------------------------------------------------
    # TRANSACTION COST
    # --------------------------------------------------------

    df["Position_Change"] = (
        df["Position"].diff().abs()
    )

    df["Strategy_Return"] = (
        df["Strategy_Return"]
        -
        df["Position_Change"] *
        TRANSACTION_COST
    )

    df = df.dropna(
        subset=["Strategy_Return"]
    ).copy()

    return df

In [23]:
# ============================================================
# Q-SCANNER V6 — PERFORMANCE METRICS
# ============================================================

def calculate_performance(df):

    returns = df["Strategy_Return"]

    cumulative_return = (
        (1 + returns).prod() - 1
    )

    # --------------------------------------------------------
    # ANNUALIZED RETURN
    # --------------------------------------------------------

    years = len(df) / 252

    if years > 0:

        cagr = (
            (1 + cumulative_return)
            ** (1 / years)
            - 1
        )

    else:
        cagr = 0

    # --------------------------------------------------------
    # VOLATILITY
    # --------------------------------------------------------

    annual_volatility = (
        returns.std() *
        np.sqrt(252)
    )

    # --------------------------------------------------------
    # SHARPE
    # --------------------------------------------------------

    if annual_volatility != 0:

        sharpe = (
            returns.mean() /
            returns.std()
        ) * np.sqrt(252)

    else:
        sharpe = 0

    # --------------------------------------------------------
    # SORTINO
    # --------------------------------------------------------

    downside = returns[
        returns < 0
    ]

    if len(downside) > 0:

        downside_std = (
            downside.std() *
            np.sqrt(252)
        )

        sortino = (
            returns.mean() *
            252
        ) / downside_std

    else:
        sortino = 0

    # --------------------------------------------------------
    # EQUITY CURVE
    # --------------------------------------------------------

    equity = (
        1 + returns
    ).cumprod()

    running_max = equity.cummax()

    drawdown = (
        equity / running_max
    ) - 1

    max_drawdown = drawdown.min()

    # --------------------------------------------------------
    # CALMAR
    # --------------------------------------------------------

    if max_drawdown != 0:

        calmar = (
            cagr /
            abs(max_drawdown)
        )

    else:
        calmar = 0

    # --------------------------------------------------------
    # WIN RATE
    # --------------------------------------------------------

    active = df[
        df["Position"] != 0
    ]

    if len(active) > 0:

        win_rate = (
            active["Strategy_Return"] > 0
        ).mean()

    else:
        win_rate = 0

    # --------------------------------------------------------
    # PROFIT FACTOR
    # --------------------------------------------------------

    gross_profit = (
        returns[returns > 0].sum()
    )

    gross_loss = abs(
        returns[returns < 0].sum()
    )

    if gross_loss > 0:

        profit_factor = (
            gross_profit /
            gross_loss
        )

    else:
        profit_factor = np.inf

    # --------------------------------------------------------
    # EXPOSURE
    # --------------------------------------------------------

    exposure = (
        (df["Position"] != 0)
        .mean()
    )

    # --------------------------------------------------------
    # TRADES
    # --------------------------------------------------------

    trades = (
        df["Position"]
        .diff()
        .abs()
        .sum()
    )

    return {
        "Strategy Return %":
            cumulative_return * 100,

        "CAGR %":
            cagr * 100,

        "Annual Volatility %":
            annual_volatility * 100,

        "Sharpe":
            sharpe,

        "Sortino":
            sortino,

        "Max Drawdown %":
            max_drawdown * 100,

        "Calmar":
            calmar,

        "Win Rate %":
            win_rate * 100,

        "Profit Factor":
            profit_factor,

        "Trades":
            int(trades),

        "Exposure %":
            exposure * 100
    }

In [24]:
# ============================================================
# Q-SCANNER V6 — WALK-FORWARD TEST
# ============================================================

all_results = []

periods = {
    "TRAIN": ("2021-01-01", "2024-01-01"),
    "VALIDATION": ("2024-01-01", "2025-01-01"),
    "OUT_OF_SAMPLE": ("2025-01-01", "2026-08-31")
}


for ticker in TICKERS:

    print("=" * 70)
    print(f"Processing {ticker}")
    print("=" * 70)

    try:

        df = prepare_v6_data(ticker)

        if df is None or df.empty:
            print("No data.")
            continue

        strategy_df = run_v6_strategy(df)

        for period_name, (
            start_date,
            end_date
        ) in periods.items():

            period_df = strategy_df.loc[
                (strategy_df.index >= start_date) &
                (strategy_df.index < end_date)
            ].copy()

            if len(period_df) < 50:
                continue

            metrics = calculate_performance(
                period_df
            )

            metrics["Ticker"] = ticker
            metrics["Period"] = period_name

            all_results.append(metrics)

    except Exception as e:

        print(
            f"{ticker}: "
            f"{type(e).__name__}: {e}"
        )


# ============================================================
# RESULTS TABLE
# ============================================================

v6_walkforward = pd.DataFrame(
    all_results
)

if not v6_walkforward.empty:

    v6_walkforward = v6_walkforward[
        [
            "Period",
            "Ticker",
            "Strategy Return %",
            "CAGR %",
            "Annual Volatility %",
            "Sharpe",
            "Sortino",
            "Max Drawdown %",
            "Calmar",
            "Win Rate %",
            "Profit Factor",
            "Trades",
            "Exposure %"
        ]
    ]

    v6_walkforward = (
        v6_walkforward
        .sort_values(
            by=[
                "Period",
                "Sharpe"
            ],
            ascending=[
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )

    print()
    print("=" * 130)
    print("Q-SCANNER V6 — WALK-FORWARD RESULTS")
    print("=" * 130)

    display(v6_walkforward)

else:

    print("No results generated.")

Processing AAPL
Processing MSFT
Processing NVDA
Processing AMZN
Processing META
Processing GOOGL
Processing TSLA

Q-SCANNER V6 — WALK-FORWARD RESULTS


,Period,Ticker,Strategy Return %,CAGR %,Annual Volatility %,Sharpe,Sortino,Max Drawdown %,Calmar,Win Rate %,Profit Factor,Trades,Exposure %
0,OUT_OF_SAMPLE,GOOGL,81.155942,43.573896,20.440591,1.870942,2.273119,-18.381935,2.370474,55.026455,1.690545,39,45.652174
1,OUT_OF_SAMPLE,MSFT,18.203295,10.715726,8.083003,1.299797,0.939255,-8.648665,1.239003,57.954545,1.618354,11,21.256039
2,OUT_OF_SAMPLE,AAPL,7.252667,4.354043,13.005014,0.392904,0.312568,-11.019980,0.395104,49.606299,1.134912,30,30.676329
3,OUT_OF_SAMPLE,NVDA,0.826954,0.502552,18.861006,0.121015,0.100043,-27.378961,0.018355,50.757576,1.034911,33,31.884058
4,OUT_OF_SAMPLE,META,-8.041046,-4.974571,11.985722,-0.365922,-0.303811,-22.388364,-0.222194,50.000000,0.890025,22,23.671498
5,OUT_OF_SAMPLE,TSLA,-19.684437,-12.491146,21.436554,-0.515637,-0.424237,-26.995927,-0.462705,45.555556,0.842509,42,21.739130
6,OUT_OF_SAMPLE,AMZN,-23.204060,-14.845834,13.755393,-1.098271,-0.792293,-27.495498,-0.539937,47.580645,0.716230,36,29.951691
7,TRAIN,AAPL,41.491209,12.316439,13.597913,0.922149,0.903382,-11.730751,1.049928,52.857143,1.284408,53,37.184595
8,TRAIN,NVDA,82.825434,22.375232,28.501668,0.845338,1.055220,-31.092963,0.719624,50.541516,1.284933,57,36.786189
9,TRAIN,META,59.596146,16.934793,21.395297,0.832376,1.067467,-14.006855,1.209036,51.171875,1.334316,49,33.997344


In [25]:
# ============================================================
# Q-SCANNER V7 — REGIME-AWARE STRATEGY
# ============================================================
#
# PURPOSE
# -------
# V7 is a controlled evolution of V6.
#
# V6 rules are preserved as the CORE SCORE. We do not add a
# large collection of indicators or optimise against the final
# OUT-OF-SAMPLE period.
#
# V7 adds only:
#   1. Trend-regime confirmation using SMA50 slope
#   2. Longer-horizon momentum confirmation
#   3. Volatility-aware position sizing
#   4. Entry/hold hysteresis to reduce unnecessary exits
#
# IMPORTANT
# ---------
# V6 OUT-OF-SAMPLE = 2025-01-01 to 2026-08-31 remains untouched.
#
# Development periods:
#   TRAIN      = 2021-01-01 to 2024-01-01
#   VALIDATION = 2024-01-01 to 2025-01-01
#   FINAL OOS  = 2025-01-01 to 2026-08-31
#
# Do NOT tune V7 using FINAL OOS results.
# ============================================================

V7_TRANSACTION_COST = 0.001
V7_TICKERS = TICKERS.copy()

V7_START_DATE = "2020-01-01"
V7_END_DATE = "2026-08-31"

V7_PERIODS = {
    "TRAIN": ("2021-01-01", "2024-01-01"),
    "VALIDATION": ("2024-01-01", "2025-01-01"),
    "OUT_OF_SAMPLE": ("2025-01-01", "2026-08-31")
}

# These are deliberately fixed before looking at the final OOS.
V7_ENTRY_SCORE = 5
V7_HOLD_SCORE = 3

# Regime thresholds
V7_MIN_ADX = 20
V7_MIN_MOMENTUM_60 = 0.0

# Position sizing:
# normal volatility       -> 100%
# elevated volatility     -> 50%
# extreme volatility      -> flat
V7_VOL_NORMAL_MULT = 1.50
V7_VOL_EXTREME_MULT = 2.50


In [26]:
# ============================================================
# Q-SCANNER V7 — ADDITIONAL FEATURES
# ============================================================

def calculate_v7_indicators(df):
    """
    Start from the existing V6 indicators and add only features
    required by the V7 regime/position-sizing hypothesis.
    """

    df = calculate_v6_indicators(df).copy()

    # SMA50 slope:
    # positive means the medium-term trend is rising.
    df["SMA50_Slope_20"] = (
        df["SMA_50"] / df["SMA_50"].shift(20) - 1
    )

    # Longer-horizon momentum.
    df["Momentum_60"] = (
        df["Close"].pct_change(60) * 100
    )

    # Rolling median ATR% used for volatility classification.
    # The current ATR% is compared with its own recent history.
    df["ATR_Pct_Median_60"] = (
        df["ATR_Pct"].rolling(60).median()
    )

    df["ATR_Vol_Ratio"] = (
        df["ATR_Pct"] /
        df["ATR_Pct_Median_60"]
    )

    df = df.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna().copy()

    return df


In [27]:
# ============================================================
# Q-SCANNER V7 — CORE SCORE + REGIME ENGINE
# ============================================================

def calculate_v7_core_score(row):
    """
    Exact V6 scoring logic, kept as the core of V7.
    """

    score = 0

    price = float(row["Close"])
    sma20 = float(row["SMA_20"])
    sma50 = float(row["SMA_50"])
    sma200 = float(row["SMA_200"])
    rsi = float(row["RSI"])
    momentum = float(row["Momentum_10"])
    adx = float(row["ADX"])

    if price > sma200:
        score += 2
    else:
        score -= 2

    if sma20 > sma50:
        score += 1
    else:
        score -= 1

    if price > sma20:
        score += 1
    else:
        score -= 1

    if 50 <= rsi < 65:
        score += 1
    elif 65 <= rsi < 70:
        score += 0.5
    elif rsi >= 70:
        score -= 1
    elif 30 <= rsi < 50:
        score -= 1
    elif rsi < 30:
        score += 0.5

    if momentum > 0:
        score += 1
    else:
        score -= 1

    if adx >= 25:
        score += 1
    elif adx < 15:
        score -= 0.5

    return score


def classify_v7_regime(row):
    """
    Transparent regime classification.

    BULL_TREND:
      Price > SMA200
      SMA50 slope positive
      ADX >= minimum threshold
      60-day momentum positive

    MIXED:
      conditions are not all aligned

    HIGH_VOL:
      ATR% is unusually high relative to its recent history
    """

    price = float(row["Close"])
    sma200 = float(row["SMA_200"])
    sma50_slope = float(row["SMA50_Slope_20"])
    adx = float(row["ADX"])
    momentum60 = float(row["Momentum_60"])
    vol_ratio = float(row["ATR_Vol_Ratio"])

    if vol_ratio >= V7_VOL_EXTREME_MULT:
        return "HIGH_VOL"

    if (
        price > sma200
        and sma50_slope > 0
        and adx >= V7_MIN_ADX
        and momentum60 > V7_MIN_MOMENTUM_60
    ):
        return "BULL_TREND"

    return "MIXED"


def v7_position_size(row, core_score, previous_position):
    """
    Long-only position sizing with hysteresis.

    Entry requires a strong V6 score and a confirmed bull regime.

    Once in a trade, the strategy can remain invested at a
    lower score (HOLD_SCORE) to reduce unnecessary whipsaw exits.

    Volatility controls size rather than changing the signal.
    """

    regime = classify_v7_regime(row)
    vol_ratio = float(row["ATR_Vol_Ratio"])

    # Hard regime exit.
    if regime != "BULL_TREND":
        return 0.0

    # Entry / continuation logic.
    if previous_position <= 0:
        if core_score < V7_ENTRY_SCORE:
            return 0.0
    else:
        if core_score < V7_HOLD_SCORE:
            return 0.0

    # Volatility-aware sizing.
    if vol_ratio >= V7_VOL_EXTREME_MULT:
        return 0.0

    if vol_ratio >= V7_VOL_NORMAL_MULT:
        return 0.5

    return 1.0


In [28]:
# ============================================================
# Q-SCANNER V7 — STRATEGY SIMULATION
# ============================================================

def run_v7_strategy(df):
    df = df.copy()

    core_scores = []
    regimes = []
    positions = []

    previous_position = 0.0

    for _, row in df.iterrows():

        core_score = calculate_v7_core_score(row)
        regime = classify_v7_regime(row)

        position = v7_position_size(
            row,
            core_score,
            previous_position
        )

        core_scores.append(core_score)
        regimes.append(regime)
        positions.append(position)

        previous_position = position

    df["V7 Core Score"] = core_scores
    df["V7 Regime"] = regimes
    df["Position"] = positions

    # Signal today -> return earned on the next trading day.
    df["Market_Return"] = (
        df["Close"].pct_change().shift(-1)
    )

    df["Strategy_Return_Gross"] = (
        df["Position"] *
        df["Market_Return"]
    )

    # Transaction cost on absolute changes in position.
    df["Position_Change"] = (
        df["Position"].diff().abs()
    )

    df["Strategy_Return"] = (
        df["Strategy_Return_Gross"]
        - df["Position_Change"] * V7_TRANSACTION_COST
    )

    return df.dropna(
        subset=["Strategy_Return"]
    ).copy()


In [29]:
# ============================================================
# Q-SCANNER V7 — BENCHMARK / DIAGNOSTIC METRICS
# ============================================================

def calculate_benchmark_metrics(period_df):
    """
    Buy & Hold metrics for the same exact period.
    """

    if period_df.empty:
        return {
            "Buy & Hold Return %": np.nan,
            "Buy & Hold CAGR %": np.nan
        }

    bh_returns = period_df["Market_Return"].fillna(0)
    bh_total = (1 + bh_returns).prod() - 1

    years = len(period_df) / 252

    if years > 0 and (1 + bh_total) > 0:
        bh_cagr = (1 + bh_total) ** (1 / years) - 1
    else:
        bh_cagr = np.nan

    return {
        "Buy & Hold Return %": bh_total * 100,
        "Buy & Hold CAGR %": bh_cagr * 100
    }


def calculate_v7_performance(df):
    """
    Same metric definitions used by V6 so V6 and V7 are comparable.
    """

    metrics = calculate_performance(
        df.rename(columns={"Strategy_Return": "Strategy_Return"})
    )

    benchmark = calculate_benchmark_metrics(df)

    metrics.update(benchmark)

    # Average position gives a more useful exposure measure
    # when V7 uses 0 / 0.5 / 1.0 sizing.
    metrics["Average Position %"] = (
        df["Position"].mean() * 100
    )

    return metrics


In [30]:
# ============================================================
# Q-SCANNER V7 — WALK-FORWARD TEST
# ============================================================
#
# FINAL OOS IS SHOWN, BUT IT MUST NOT BE USED FOR OPTIMISATION.
# ============================================================

v7_all_results = []
v7_data_cache = {}

for ticker in V7_TICKERS:

    print("=" * 75)
    print(f"Processing V7: {ticker}")
    print("=" * 75)

    try:

        raw = yf.download(
            ticker,
            start=V7_START_DATE,
            end=V7_END_DATE,
            auto_adjust=True,
            progress=False
        )

        if raw is None or raw.empty:
            print(f"{ticker}: No data.")
            continue

        df = calculate_v7_indicators(raw)

        if df.empty:
            print(f"{ticker}: Not enough data.")
            continue

        strategy_df = run_v7_strategy(df)
        v7_data_cache[ticker] = strategy_df

        for period_name, (start_date, end_date) in V7_PERIODS.items():

            period_df = strategy_df.loc[
                (strategy_df.index >= start_date) &
                (strategy_df.index < end_date)
            ].copy()

            if len(period_df) < 50:
                continue

            metrics = calculate_v7_performance(period_df)

            metrics["Ticker"] = ticker
            metrics["Period"] = period_name

            v7_all_results.append(metrics)

    except Exception as e:
        print(
            f"{ticker}: "
            f"{type(e).__name__}: {e}"
        )


v7_walkforward = pd.DataFrame(v7_all_results)

if not v7_walkforward.empty:

    v7_walkforward = v7_walkforward[
        [
            "Period",
            "Ticker",
            "Strategy Return %",
            "Buy & Hold Return %",
            "CAGR %",
            "Buy & Hold CAGR %",
            "Annual Volatility %",
            "Sharpe",
            "Sortino",
            "Max Drawdown %",
            "Calmar",
            "Win Rate %",
            "Profit Factor",
            "Trades",
            "Exposure %",
            "Average Position %"
        ]
    ].sort_values(
        ["Period", "Sharpe"],
        ascending=[True, False]
    ).reset_index(drop=True)

    print()
    print("=" * 145)
    print("Q-SCANNER V7 — WALK-FORWARD RESULTS")
    print("=" * 145)

    display(v7_walkforward)

else:
    print("No V7 results generated.")


Processing V7: AAPL
Processing V7: MSFT
Processing V7: NVDA
Processing V7: AMZN
Processing V7: META
Processing V7: GOOGL
Processing V7: TSLA

Q-SCANNER V7 — WALK-FORWARD RESULTS


,Period,Ticker,Strategy Return %,Buy & Hold Return %,CAGR %,Buy & Hold CAGR %,Annual Volatility %,Sharpe,Sortino,Max Drawdown %,Calmar,Win Rate %,Profit Factor,Trades,Exposure %,Average Position %
0,OUT_OF_SAMPLE,GOOGL,59.419549,83.920344,32.826943,44.903538,19.652405,1.541276,2.048712,-11.301266,2.904714,52.542373,1.545876,15,42.753623,42.753623
1,OUT_OF_SAMPLE,MSFT,10.496847,24.372925,6.264178,14.198188,7.763816,0.821294,0.564867,-12.837494,0.487960,56.470588,1.359931,7,20.531401,20.531401
2,OUT_OF_SAMPLE,NVDA,9.236080,57.526523,5.524495,31.864633,15.373914,0.426483,0.326189,-19.112520,0.289051,52.000000,1.148105,12,24.154589,24.154589
3,OUT_OF_SAMPLE,META,5.889967,-3.064375,3.544972,-1.876617,9.635151,0.409585,0.284758,-12.043143,0.294356,55.882353,1.166429,8,16.425121,16.425121
4,OUT_OF_SAMPLE,AAPL,1.262457,32.056618,0.766565,18.442125,12.868325,0.123942,0.088720,-9.256313,0.082815,49.586777,1.042688,12,29.227053,29.106280
5,OUT_OF_SAMPLE,AMZN,-11.261568,20.983558,-7.014368,12.293643,10.527466,-0.636992,-0.297216,-14.633190,-0.479346,52.238806,0.759256,10,16.183575,16.183575
6,OUT_OF_SAMPLE,TSLA,-29.145182,-8.049462,-18.918738,-4.979864,19.897155,-0.951159,-0.490219,-29.270003,-0.646352,40.983607,0.651937,14,14.734300,14.734300
7,TRAIN,NVDA,112.492293,256.765031,28.996241,53.672311,28.372541,1.032799,1.315995,-20.360325,1.424154,50.793651,1.397676,24,33.780161,32.841823
8,TRAIN,META,59.619391,37.613267,17.112340,11.388325,20.568226,0.864598,1.119964,-15.366454,1.113617,48.372093,1.404902,21,28.820375,28.820375
9,TRAIN,AAPL,24.289841,44.333839,7.621864,13.196975,11.879384,0.677648,0.591218,-11.403126,0.668401,50.446429,1.229290,27,30.026810,29.892761


In [31]:
# ============================================================
# Q-SCANNER V7 — PERIOD SUMMARY
# ============================================================

if not v7_walkforward.empty:

    v7_summary = (
        v7_walkforward
        .groupby("Period")
        .agg(
            Mean_Return=("Strategy Return %", "mean"),
            Median_Return=("Strategy Return %", "median"),
            Mean_CAGR=("CAGR %", "mean"),
            Mean_Sharpe=("Sharpe", "mean"),
            Median_Sharpe=("Sharpe", "median"),
            Mean_Max_DD=("Max Drawdown %", "mean"),
            Mean_PF=("Profit Factor", "mean"),
            Mean_Exposure=("Exposure %", "mean"),
            Mean_BH_Return=("Buy & Hold Return %", "mean")
        )
        .reset_index()
    )

    print("=" * 120)
    print("Q-SCANNER V7 — PERIOD SUMMARY")
    print("=" * 120)
    display(v7_summary)


Q-SCANNER V7 — PERIOD SUMMARY


,Period,Mean_Return,Median_Return,Mean_CAGR,Mean_Sharpe,Median_Sharpe,Mean_Max_DD,Mean_PF,Mean_Exposure,Mean_BH_Return
0,OUT_OF_SAMPLE,6.556879,5.889967,3.284864,0.247776,0.409585,-15.493419,1.096317,23.429952,29.678019
1,TRAIN,30.547574,24.289841,8.294266,0.395305,0.617234,-17.710864,1.177371,25.698966,64.977620
2,VALIDATION,18.641142,13.781892,18.641142,0.686407,0.755768,-14.808809,1.245062,35.374150,63.398898


In [32]:
# ============================================================
# Q-SCANNER V7 — V6 VS V7 COMPARISON
# ============================================================
#
# This compares V7 with the previously generated V6 results.
# The comparison is diagnostic only.
# ============================================================

if (
    "v6_walkforward" in globals()
    and not v6_walkforward.empty
    and not v7_walkforward.empty
):

    v6_cmp = v6_walkforward.copy()

    v6_cmp["Version"] = "V6"
    v7_cmp = v7_walkforward.copy()
    v7_cmp["Version"] = "V7"

    comparison_cols = [
        "Period",
        "Ticker",
        "Version",
        "Strategy Return %",
        "CAGR %",
        "Sharpe",
        "Sortino",
        "Max Drawdown %",
        "Profit Factor",
        "Trades",
        "Exposure %"
    ]

    comparison = pd.concat(
        [
            v6_cmp[comparison_cols],
            v7_cmp[comparison_cols]
        ],
        ignore_index=True
    ).sort_values(
        ["Period", "Ticker", "Version"]
    ).reset_index(drop=True)

    print("=" * 130)
    print("Q-SCANNER — V6 VS V7")
    print("=" * 130)
    display(comparison)

else:
    print(
        "V6 walk-forward results are not available in this kernel. "
        "Run the V6 walk-forward cell first."
    )


Q-SCANNER — V6 VS V7


,Period,Ticker,Version,Strategy Return %,CAGR %,Sharpe,Sortino,Max Drawdown %,Profit Factor,Trades,Exposure %
0,OUT_OF_SAMPLE,AAPL,V6,7.252667,4.354043,0.392904,0.312568,-11.019980,1.134912,30,30.676329
1,OUT_OF_SAMPLE,AAPL,V7,1.262457,0.766565,0.123942,0.088720,-9.256313,1.042688,12,29.227053
2,OUT_OF_SAMPLE,AMZN,V6,-23.204060,-14.845834,-1.098271,-0.792293,-27.495498,0.716230,36,29.951691
3,OUT_OF_SAMPLE,AMZN,V7,-11.261568,-7.014368,-0.636992,-0.297216,-14.633190,0.759256,10,16.183575
4,OUT_OF_SAMPLE,GOOGL,V6,81.155942,43.573896,1.870942,2.273119,-18.381935,1.690545,39,45.652174
5,OUT_OF_SAMPLE,GOOGL,V7,59.419549,32.826943,1.541276,2.048712,-11.301266,1.545876,15,42.753623
6,OUT_OF_SAMPLE,META,V6,-8.041046,-4.974571,-0.365922,-0.303811,-22.388364,0.890025,22,23.671498
7,OUT_OF_SAMPLE,META,V7,5.889967,3.544972,0.409585,0.284758,-12.043143,1.166429,8,16.425121
8,OUT_OF_SAMPLE,MSFT,V6,18.203295,10.715726,1.299797,0.939255,-8.648665,1.618354,11,21.256039
9,OUT_OF_SAMPLE,MSFT,V7,10.496847,6.264178,0.821294,0.564867,-12.837494,1.359931,7,20.531401


In [33]:
# ============================================================
# Q-SCANNER V7 — PARAMETER STABILITY TEST
# ============================================================
#
# IMPORTANT:
#   This test is NOT used to optimise the final OOS.
#
# We perturb the main regime thresholds around their fixed
# baseline values. A robust strategy should not collapse when
# parameters move slightly.
#
# The stability test is run on TRAIN + VALIDATION only.
# ============================================================

def run_v7_with_params(df, entry_score, hold_score, min_adx):
    """
    Same V7 architecture with only small threshold perturbations.
    """

    temp = df.copy()

    positions = []
    previous_position = 0.0

    for _, row in temp.iterrows():

        score = calculate_v7_core_score(row)

        price = float(row["Close"])
        sma200 = float(row["SMA_200"])
        sma50_slope = float(row["SMA50_Slope_20"])
        adx = float(row["ADX"])
        momentum60 = float(row["Momentum_60"])
        vol_ratio = float(row["ATR_Vol_Ratio"])

        bull = (
            price > sma200
            and sma50_slope > 0
            and adx >= min_adx
            and momentum60 > 0
        )

        if not bull:
            position = 0.0

        elif previous_position <= 0:
            if score < entry_score:
                position = 0.0
            elif vol_ratio >= V7_VOL_EXTREME_MULT:
                position = 0.0
            elif vol_ratio >= V7_VOL_NORMAL_MULT:
                position = 0.5
            else:
                position = 1.0

        else:
            if score < hold_score:
                position = 0.0
            elif vol_ratio >= V7_VOL_EXTREME_MULT:
                position = 0.0
            elif vol_ratio >= V7_VOL_NORMAL_MULT:
                position = 0.5
            else:
                position = 1.0

        positions.append(position)
        previous_position = position

    temp["Position"] = positions
    temp["Market_Return"] = (
        temp["Close"].pct_change().shift(-1)
    )
    temp["Strategy_Return"] = (
        temp["Position"] * temp["Market_Return"]
        - temp["Position"].diff().abs() * V7_TRANSACTION_COST
    )

    return temp.dropna(
        subset=["Strategy_Return"]
    ).copy()


stability_results = []

for ticker, df in v7_data_cache.items():

    development_df = df.loc[
        df.index < "2025-01-01"
    ].copy()

    if development_df.empty:
        continue

    for entry in [4.5, 5.0, 5.5]:
        for hold in [2.5, 3.0, 3.5]:
            for adx_threshold in [18, 20, 22]:

                test_df = run_v7_with_params(
                    development_df,
                    entry,
                    hold,
                    adx_threshold
                )

                if len(test_df) < 100:
                    continue

                m = calculate_performance(test_df)

                stability_results.append({
                    "Ticker": ticker,
                    "Entry Score": entry,
                    "Hold Score": hold,
                    "Min ADX": adx_threshold,
                    "Return %": m["Strategy Return %"],
                    "CAGR %": m["CAGR %"],
                    "Sharpe": m["Sharpe"],
                    "Max DD %": m["Max Drawdown %"],
                    "Profit Factor": m["Profit Factor"],
                    "Trades": m["Trades"]
                })

v7_stability = pd.DataFrame(stability_results)

if not v7_stability.empty:

    print("=" * 130)
    print("Q-SCANNER V7 — PARAMETER STABILITY")
    print("=" * 130)

    display(
        v7_stability.sort_values(
            ["Sharpe", "Profit Factor"],
            ascending=[False, False]
        ).head(30)
    )

else:
    print("No stability results generated.")


Q-SCANNER V7 — PARAMETER STABILITY


,Ticker,Entry Score,Hold Score,Min ADX,Return %,CAGR %,Sharpe,Max DD %,Profit Factor,Trades
55,NVDA,4.5,2.5,20,276.177512,39.823870,1.272884,-16.674872,1.472463,37
58,NVDA,4.5,3.0,20,276.177512,39.823870,1.272884,-16.674872,1.472463,37
56,NVDA,4.5,2.5,22,245.843113,36.880919,1.231545,-16.674872,1.487470,35
59,NVDA,4.5,3.0,22,245.843113,36.880919,1.231545,-16.674872,1.487470,35
64,NVDA,5.0,2.5,20,245.156693,36.812130,1.203964,-20.360325,1.444509,37
67,NVDA,5.0,3.0,20,245.156693,36.812130,1.203964,-20.360325,1.444509,37
54,NVDA,4.5,2.5,18,244.094214,36.705454,1.173850,-19.542798,1.405240,41
57,NVDA,4.5,3.0,18,244.094214,36.705454,1.173850,-19.542798,1.405240,41
65,NVDA,5.0,2.5,22,217.323767,33.932569,1.160313,-16.674872,1.456653,35
68,NVDA,5.0,3.0,22,217.323767,33.932569,1.160313,-16.674872,1.456653,35


In [34]:
# ============================================================
# Q-SCANNER V7 — FINAL OOS LOCK
# ============================================================
#
# This is the number we care about after development.
#
# DO NOT change V7 parameters after seeing this table.
# If we modify V7 from this point, the 2025-2026 period is no
# longer a clean final OOS test and we must reserve a new period.
# ============================================================

if not v7_walkforward.empty:

    final_oos = v7_walkforward[
        v7_walkforward["Period"] == "OUT_OF_SAMPLE"
    ].copy()

    final_oos = final_oos.sort_values(
        "Sharpe",
        ascending=False
    ).reset_index(drop=True)

    print("=" * 145)
    print("Q-SCANNER V7 — FINAL OUT-OF-SAMPLE (LOCKED)")
    print("=" * 145)

    display(final_oos)

    print()
    print("RULE: Do not optimise V7 using these results.")


Q-SCANNER V7 — FINAL OUT-OF-SAMPLE (LOCKED)


,Period,Ticker,Strategy Return %,Buy & Hold Return %,CAGR %,Buy & Hold CAGR %,Annual Volatility %,Sharpe,Sortino,Max Drawdown %,Calmar,Win Rate %,Profit Factor,Trades,Exposure %,Average Position %
0,OUT_OF_SAMPLE,GOOGL,59.419549,83.920344,32.826943,44.903538,19.652405,1.541276,2.048712,-11.301266,2.904714,52.542373,1.545876,15,42.753623,42.753623
1,OUT_OF_SAMPLE,MSFT,10.496847,24.372925,6.264178,14.198188,7.763816,0.821294,0.564867,-12.837494,0.487960,56.470588,1.359931,7,20.531401,20.531401
2,OUT_OF_SAMPLE,NVDA,9.236080,57.526523,5.524495,31.864633,15.373914,0.426483,0.326189,-19.112520,0.289051,52.000000,1.148105,12,24.154589,24.154589
3,OUT_OF_SAMPLE,META,5.889967,-3.064375,3.544972,-1.876617,9.635151,0.409585,0.284758,-12.043143,0.294356,55.882353,1.166429,8,16.425121,16.425121
4,OUT_OF_SAMPLE,AAPL,1.262457,32.056618,0.766565,18.442125,12.868325,0.123942,0.088720,-9.256313,0.082815,49.586777,1.042688,12,29.227053,29.106280
5,OUT_OF_SAMPLE,AMZN,-11.261568,20.983558,-7.014368,12.293643,10.527466,-0.636992,-0.297216,-14.633190,-0.479346,52.238806,0.759256,10,16.183575,16.183575
6,OUT_OF_SAMPLE,TSLA,-29.145182,-8.049462,-18.918738,-4.979864,19.897155,-0.951159,-0.490219,-29.270003,-0.646352,40.983607,0.651937,14,14.734300,14.734300



RULE: Do not optimise V7 using these results.


In [35]:
# ============================================================
# Q-SCANNER V7 — FINAL OOS LOCK
# ============================================================
#
# This is the number we care about after development.
#
# DO NOT change V7 parameters after seeing this table.
# If we modify V7 from this point, the 2025-2026 period is no
# longer a clean final OOS test and we must reserve a new period.
# ============================================================

if not v7_walkforward.empty:

    final_oos = v7_walkforward[
        v7_walkforward["Period"] == "OUT_OF_SAMPLE"
    ].copy()

    final_oos = final_oos.sort_values(
        "Sharpe",
        ascending=False
    ).reset_index(drop=True)

    print("=" * 145)
    print("Q-SCANNER V7 — FINAL OUT-OF-SAMPLE (LOCKED)")
    print("=" * 145)

    display(final_oos)

    print()
    print("RULE: Do not optimise V7 using these results.")


Q-SCANNER V7 — FINAL OUT-OF-SAMPLE (LOCKED)


,Period,Ticker,Strategy Return %,Buy & Hold Return %,CAGR %,Buy & Hold CAGR %,Annual Volatility %,Sharpe,Sortino,Max Drawdown %,Calmar,Win Rate %,Profit Factor,Trades,Exposure %,Average Position %
0,OUT_OF_SAMPLE,GOOGL,59.419549,83.920344,32.826943,44.903538,19.652405,1.541276,2.048712,-11.301266,2.904714,52.542373,1.545876,15,42.753623,42.753623
1,OUT_OF_SAMPLE,MSFT,10.496847,24.372925,6.264178,14.198188,7.763816,0.821294,0.564867,-12.837494,0.487960,56.470588,1.359931,7,20.531401,20.531401
2,OUT_OF_SAMPLE,NVDA,9.236080,57.526523,5.524495,31.864633,15.373914,0.426483,0.326189,-19.112520,0.289051,52.000000,1.148105,12,24.154589,24.154589
3,OUT_OF_SAMPLE,META,5.889967,-3.064375,3.544972,-1.876617,9.635151,0.409585,0.284758,-12.043143,0.294356,55.882353,1.166429,8,16.425121,16.425121
4,OUT_OF_SAMPLE,AAPL,1.262457,32.056618,0.766565,18.442125,12.868325,0.123942,0.088720,-9.256313,0.082815,49.586777,1.042688,12,29.227053,29.106280
5,OUT_OF_SAMPLE,AMZN,-11.261568,20.983558,-7.014368,12.293643,10.527466,-0.636992,-0.297216,-14.633190,-0.479346,52.238806,0.759256,10,16.183575,16.183575
6,OUT_OF_SAMPLE,TSLA,-29.145182,-8.049462,-18.918738,-4.979864,19.897155,-0.951159,-0.490219,-29.270003,-0.646352,40.983607,0.651937,14,14.734300,14.734300



RULE: Do not optimise V7 using these results.


In [36]:
# ============================================================
# Q-SCANNER V7 — STAGE 2
# FINAL OOS CROSS-SECTIONAL DIAGNOSTIC
# ============================================================
#
# IMPORTANT:
# This cell does NOT modify V7.
# It only analyses the already-locked final_oos results.
# ============================================================

if not final_oos.empty:

    print("=" * 145)
    print("Q-SCANNER V7 — FINAL OOS CROSS-SECTIONAL DIAGNOSTIC")
    print("=" * 145)

    # --------------------------------------------------------
    # 1. Basic portfolio-style statistics
    # --------------------------------------------------------

    mean_return = final_oos["Strategy Return %"].mean()
    median_return = final_oos["Strategy Return %"].median()

    mean_sharpe = final_oos["Sharpe"].mean()
    median_sharpe = final_oos["Sharpe"].median()

    mean_profit_factor = final_oos["Profit Factor"].mean()
    median_profit_factor = final_oos["Profit Factor"].median()

    mean_drawdown = final_oos["Max Drawdown %"].mean()

    positive_tickers = (final_oos["Strategy Return %"] > 0).sum()
    profitable_pf = (final_oos["Profit Factor"] > 1).sum()
    positive_sharpe = (final_oos["Sharpe"] > 0).sum()

    print()
    print("RETURN")
    print("-" * 60)
    print(f"Mean OOS Return:       {mean_return:.2f}%")
    print(f"Median OOS Return:     {median_return:.2f}%")

    print()
    print("RISK-ADJUSTED PERFORMANCE")
    print("-" * 60)
    print(f"Mean Sharpe:            {mean_sharpe:.2f}")
    print(f"Median Sharpe:          {median_sharpe:.2f}")
    print(f"Mean Profit Factor:     {mean_profit_factor:.2f}")
    print(f"Median Profit Factor:   {median_profit_factor:.2f}")
    print(f"Mean Max Drawdown:      {mean_drawdown:.2f}%")

    print()
    print("BREADTH OF EDGE")
    print("-" * 60)
    print(
        f"Positive-return tickers: "
        f"{positive_tickers}/{len(final_oos)}"
    )

    print(
        f"Profit Factor > 1:      "
        f"{profitable_pf}/{len(final_oos)}"
    )

    print(
        f"Positive Sharpe:        "
        f"{positive_sharpe}/{len(final_oos)}"
    )

    # --------------------------------------------------------
    # 2. Rank the tickers
    # --------------------------------------------------------

    diagnostic = final_oos[
        [
            "Ticker",
            "Strategy Return %",
            "CAGR %",
            "Sharpe",
            "Sortino",
            "Max Drawdown %",
            "Calmar",
            "Win Rate %",
            "Profit Factor",
            "Trades",
            "Exposure %"
        ]
    ].copy()

    diagnostic["Score"] = (
        diagnostic["Sharpe"].rank(pct=True)
        + diagnostic["Profit Factor"].rank(pct=True)
        + diagnostic["CAGR %"].rank(pct=True)
    ) / 3

    diagnostic = diagnostic.sort_values(
        "Score",
        ascending=False
    ).reset_index(drop=True)

    print()
    print("=" * 145)
    print("OOS TICKER RANKING")
    print("=" * 145)

    display(diagnostic)

    # --------------------------------------------------------
    # 3. Important warning
    # --------------------------------------------------------

    print()
    print("=" * 145)
    print("IMPORTANT")
    print("=" * 145)

    print(
        "These statistics describe the locked OOS results only."
    )

    print(
        "They are NOT a true portfolio equity curve."
    )

    print(
        "A true portfolio Sharpe, Sortino, CAGR and max drawdown "
        "require the underlying time-series equity/return data."
    )

    print()
    print(
        "NEXT STAGE: reconstruct the OOS portfolio equity curve "
        "from the underlying backtest data."
    )

Q-SCANNER V7 — FINAL OOS CROSS-SECTIONAL DIAGNOSTIC

RETURN
------------------------------------------------------------
Mean OOS Return:       6.56%
Median OOS Return:     5.89%

RISK-ADJUSTED PERFORMANCE
------------------------------------------------------------
Mean Sharpe:            0.25
Median Sharpe:          0.41
Mean Profit Factor:     1.10
Median Profit Factor:   1.15
Mean Max Drawdown:      -15.49%

BREADTH OF EDGE
------------------------------------------------------------
Positive-return tickers: 5/7
Profit Factor > 1:      5/7
Positive Sharpe:        5/7

OOS TICKER RANKING


,Ticker,Strategy Return %,CAGR %,Sharpe,Sortino,Max Drawdown %,Calmar,Win Rate %,Profit Factor,Trades,Exposure %,Score
0,GOOGL,59.419549,32.826943,1.541276,2.048712,-11.301266,2.904714,52.542373,1.545876,15,42.753623,1.000000
1,MSFT,10.496847,6.264178,0.821294,0.564867,-12.837494,0.487960,56.470588,1.359931,7,20.531401,0.857143
2,NVDA,9.236080,5.524495,0.426483,0.326189,-19.112520,0.289051,52.000000,1.148105,12,24.154589,0.666667
3,META,5.889967,3.544972,0.409585,0.284758,-12.043143,0.294356,55.882353,1.166429,8,16.425121,0.619048
4,AAPL,1.262457,0.766565,0.123942,0.088720,-9.256313,0.082815,49.586777,1.042688,12,29.227053,0.428571
5,AMZN,-11.261568,-7.014368,-0.636992,-0.297216,-14.633190,-0.479346,52.238806,0.759256,10,16.183575,0.285714
6,TSLA,-29.145182,-18.918738,-0.951159,-0.490219,-29.270003,-0.646352,40.983607,0.651937,14,14.734300,0.142857



IMPORTANT
These statistics describe the locked OOS results only.
They are NOT a true portfolio equity curve.
A true portfolio Sharpe, Sortino, CAGR and max drawdown require the underlying time-series equity/return data.

NEXT STAGE: reconstruct the OOS portfolio equity curve from the underlying backtest data.


In [37]:
# ============================================================
# Q-SCANNER V7 — STAGE 3A
# LOCATE UNDERLYING OOS BACKTEST DATA
# ============================================================

print("V7 WALK-FORWARD OBJECT:")
print(type(v7_walkforward))

print()
print("V7 WALK-FORWARD COLUMNS:")
print(list(v7_walkforward.columns))

print()
print("AVAILABLE VARIABLES RELATED TO BACKTEST / EQUITY / RETURNS:")
for name in sorted(globals()):
    name_lower = name.lower()
    if any(x in name_lower for x in [
        "equity",
        "return",
        "backtest",
        "portfolio",
        "trade",
        "oos",
        "walkforward"
    ]):
        print(name)

V7 WALK-FORWARD OBJECT:
<class 'pandas.DataFrame'>

V7 WALK-FORWARD COLUMNS:
['Period', 'Ticker', 'Strategy Return %', 'Buy & Hold Return %', 'CAGR %', 'Buy & Hold CAGR %', 'Annual Volatility %', 'Sharpe', 'Sortino', 'Max Drawdown %', 'Calmar', 'Win Rate %', 'Profit Factor', 'Trades', 'Exposure %', 'Average Position %']

AVAILABLE VARIABLES RELATED TO BACKTEST / EQUITY / RETURNS:
BACKTEST_PERIOD
BACKTEST_TICKERS
backtest_df
backtest_results
backtest_ticker
backtest_v4
backtest_v4_df
backtest_v5
final_oos
mean_return
median_return
spy_return
strategy_return
v6_walkforward
v7_walkforward


In [38]:
# ============================================================
# Q-SCANNER V7 — OOS EQUITY RECONSTRUCTION
# STEP 1: INSPECT UNDERLYING BACKTEST OBJECTS
# ============================================================

print("=" * 100)
print("Q-SCANNER V7 — OOS EQUITY RECONSTRUCTION")
print("=" * 100)

print("\nAVAILABLE V7/BACKTEST OBJECTS:")
for name in [
    "backtest_df",
    "backtest_results",
    "backtest_v7_df",
    "backtest_v7",
    "v7_walkforward",
    "final_oos"
]:
    if name in globals():
        obj = globals()[name]

        print(f"\n{name}")
        print("-" * 80)
        print("Type:", type(obj))

        if hasattr(obj, "shape"):
            print("Shape:", obj.shape)

        if hasattr(obj, "columns"):
            print("Columns:")
            print(list(obj.columns))

        if hasattr(obj, "index"):
            print("Index type:", type(obj.index))

print("\n" + "=" * 100)
print("END OF INSPECTION")
print("=" * 100)

Q-SCANNER V7 — OOS EQUITY RECONSTRUCTION

AVAILABLE V7/BACKTEST OBJECTS:

backtest_df
--------------------------------------------------------------------------------
Type: <class 'pandas.DataFrame'>
Shape: (7, 6)
Columns:
['Ticker', 'Strategy Return %', 'Buy & Hold %', 'Win Rate %', 'Max Drawdown %', 'Signal Days']
Index type: <class 'pandas.RangeIndex'>

backtest_results
--------------------------------------------------------------------------------
Type: <class 'list'>
Index type: <class 'builtin_function_or_method'>

v7_walkforward
--------------------------------------------------------------------------------
Type: <class 'pandas.DataFrame'>
Shape: (21, 16)
Columns:
['Period', 'Ticker', 'Strategy Return %', 'Buy & Hold Return %', 'CAGR %', 'Buy & Hold CAGR %', 'Annual Volatility %', 'Sharpe', 'Sortino', 'Max Drawdown %', 'Calmar', 'Win Rate %', 'Profit Factor', 'Trades', 'Exposure %', 'Average Position %']
Index type: <class 'pandas.RangeIndex'>

final_oos
----------------------

In [39]:
# ============================================================
# Q-SCANNER V7 — OOS EQUITY RECONSTRUCTION
# STEP 2: INSPECT backtest_results
# ============================================================

print("=" * 100)
print("INSPECTING backtest_results")
print("=" * 100)

print("\nType:", type(backtest_results))
print("Length:", len(backtest_results))

for i, item in enumerate(backtest_results[:10]):

    print("\n" + "-" * 80)
    print(f"ITEM {i}")
    print("-" * 80)

    print("Type:", type(item))

    if isinstance(item, dict):
        print("Dictionary keys:")
        print(list(item.keys()))

        for key, value in item.items():
            print(f"  {key}: {type(value)}", end="")

            if hasattr(value, "shape"):
                print(f" | shape={value.shape}")
            elif isinstance(value, (list, tuple)):
                print(f" | length={len(value)}")
            else:
                print()

    elif hasattr(item, "shape"):
        print("Shape:", item.shape)

        if hasattr(item, "columns"):
            print("Columns:")
            print(list(item.columns))

        print("\nFirst rows:")
        display(item.head())

    else:
        print("Value:", item)

print("\n" + "=" * 100)
print("END OF backtest_results INSPECTION")
print("=" * 100)

INSPECTING backtest_results

Type: <class 'list'>
Length: 7

--------------------------------------------------------------------------------
ITEM 0
--------------------------------------------------------------------------------
Type: <class 'dict'>
Dictionary keys:
['Ticker', 'Strategy Return %', 'Buy & Hold %', 'Win Rate %', 'Max Drawdown %', 'Signal Days']
  Ticker: <class 'str'>
  Strategy Return %: <class 'numpy.float64'> | shape=()
  Buy & Hold %: <class 'numpy.float64'> | shape=()
  Win Rate %: <class 'numpy.float64'> | shape=()
  Max Drawdown %: <class 'numpy.float64'> | shape=()
  Signal Days: <class 'int'>

--------------------------------------------------------------------------------
ITEM 1
--------------------------------------------------------------------------------
Type: <class 'dict'>
Dictionary keys:
['Ticker', 'Strategy Return %', 'Buy & Hold %', 'Win Rate %', 'Max Drawdown %', 'Signal Days']
  Ticker: <class 'str'>
  Strategy Return %: <class 'numpy.float64'> | s

In [44]:
# ============================================================
# Q-SCANNER V7 — OOS PORTFOLIO EQUITY RECONSTRUCTION
# ============================================================

print("=" * 100)
print("Q-SCANNER V7 — OOS PORTFOLIO EQUITY RECONSTRUCTION")
print("=" * 100)

# Inspect the available backtest objects and their structures
objects_to_check = {
    "backtest_df": backtest_df,
    "backtest_results": backtest_results,
    "v7_walkforward": v7_walkforward,
    "final_oos": final_oos
}

for name, obj in objects_to_check.items():
    print(f"\n{name}")
    print("-" * 80)
    print("Type:", type(obj))

    if hasattr(obj, "shape"):
        print("Shape:", obj.shape)

    if hasattr(obj, "columns"):
        print("Columns:", list(obj.columns))

    if hasattr(obj, "index"):
        print("Index type:", type(obj.index))

Q-SCANNER V7 — OOS PORTFOLIO EQUITY RECONSTRUCTION

backtest_df
--------------------------------------------------------------------------------
Type: <class 'pandas.DataFrame'>
Shape: (7, 6)
Columns: ['Ticker', 'Strategy Return %', 'Buy & Hold %', 'Win Rate %', 'Max Drawdown %', 'Signal Days']
Index type: <class 'pandas.RangeIndex'>

backtest_results
--------------------------------------------------------------------------------
Type: <class 'list'>
Index type: <class 'builtin_function_or_method'>

v7_walkforward
--------------------------------------------------------------------------------
Type: <class 'pandas.DataFrame'>
Shape: (21, 16)
Columns: ['Period', 'Ticker', 'Strategy Return %', 'Buy & Hold Return %', 'CAGR %', 'Buy & Hold CAGR %', 'Annual Volatility %', 'Sharpe', 'Sortino', 'Max Drawdown %', 'Calmar', 'Win Rate %', 'Profit Factor', 'Trades', 'Exposure %', 'Average Position %']
Index type: <class 'pandas.RangeIndex'>

final_oos
--------------------------------------------

In [45]:
# ============================================================
# Q-SCANNER V7 — STAGE 1
# LOCATE UNDERLYING TIME-SERIES / EQUITY DATA
# ============================================================

print("=" * 100)
print("Q-SCANNER V7 — SEARCHING FOR UNDERLYING OOS TIME-SERIES DATA")
print("=" * 100)

import pandas as pd
import numpy as np

candidates = []

# IMPORTANT:
# Convert globals() to a list first so the dictionary can safely
# change while we inspect objects.
for name, obj in list(globals().items()):

    if name.startswith("_"):
        continue

    try:

        if isinstance(obj, pd.DataFrame):

            cols = [str(c).lower() for c in obj.columns]
            idx = obj.index

            has_date_index = (
                isinstance(idx, pd.DatetimeIndex)
                or "date" in str(idx.name).lower()
                or "time" in str(idx.name).lower()
            )

            relevant_columns = [
                c for c in cols
                if any(key in c for key in [
                    "return",
                    "equity",
                    "portfolio",
                    "strategy",
                    "signal",
                    "position",
                    "trade",
                    "price",
                    "close",
                    "pnl"
                ])
            ]

            if has_date_index or relevant_columns:

                candidates.append({
                    "Variable": name,
                    "Shape": obj.shape,
                    "Index": type(obj.index).__name__,
                    "Index Name": obj.index.name,
                    "Relevant Columns": relevant_columns,
                    "Columns": list(obj.columns)
                })

    except Exception:
        pass


if candidates:

    candidates_df = pd.DataFrame(candidates)

    print()
    print("POTENTIAL TIME-SERIES / BACKTEST DATA FOUND:")
    print()

    display(candidates_df)

else:

    print()
    print("NO OBVIOUS TIME-SERIES DATA FOUND.")
    print()
    print("We will inspect the underlying objects more deeply before")
    print("attempting to reconstruct the OOS equity curve.")


print()
print("=" * 100)
print("END OF STAGE 1")
print("=" * 100)

Q-SCANNER V7 — SEARCHING FOR UNDERLYING OOS TIME-SERIES DATA

POTENTIAL TIME-SERIES / BACKTEST DATA FOUND:



,Variable,Shape,Index,Index Name,Relevant Columns,Columns
0,data,"(974, 19)",DatetimeIndex,Date,"[close, v6 signal]","[Close, High, Low, Open, Volume, SMA_20, SMA_5..."
1,signal,"(974, 3)",DatetimeIndex,Date,[v6 signal],"[V6 Score, V6 Signal, V6 Reasons]"
2,scanner_results,"(7, 17)",RangeIndex,NaN,"[close, signal, signal strength]","[Close, High, Low, Open, Volume, SMA_20, SMA_5..."
3,scanner_v2,"(7, 12)",RangeIndex,NaN,"[price, price trend, signal]","[Ticker, Price, SMA_20, SMA_50, RSI, Price Tre..."
4,base,"(7, 9)",RangeIndex,NaN,"[price, price trend]","[Ticker, Price, SMA_20, SMA_50, RSI, Price Tre..."
5,signal_output,"(7, 4)",RangeIndex,NaN,[new signal],"[New Score, Confidence %, New Signal, Reasons]"
6,scanner_v3,"(7, 13)",RangeIndex,NaN,"[price, price trend, new signal]","[Ticker, Price, SMA_20, SMA_50, RSI, Price Tre..."
7,backtest_df,"(7, 6)",RangeIndex,NaN,"[strategy return %, signal days]","[Ticker, Strategy Return %, Buy & Hold %, Win ..."
8,scanned,"(1124, 15)",DatetimeIndex,Date,"[close, signal]","[Close, High, Low, Open, Volume, SMA_20, SMA_5..."
9,backtest_v4_df,"(7, 11)",RangeIndex,NaN,"[strategy return %, strategy cagr %, trades]","[Ticker, Strategy Return %, Buy & Hold %, Stra..."



END OF STAGE 1


In [43]:
import numpy as np
import pandas as pd

print("=" * 80)
print("Q-SCANNER V7 — OOS TIME-SERIES EXTRACTION")
print("=" * 80)

# Snapshot globals first — prevents "dictionary changed size during iteration"
global_items = list(globals().items())

candidates = []

for name, obj in global_items:
    try:
        if isinstance(obj, pd.DataFrame):
            cols = [str(c).lower() for c in obj.columns]
            idx = obj.index

            relevant = any(
                x in " ".join(cols)
                for x in [
                    "close",
                    "return",
                    "signal",
                    "position",
                    "strategy"
                ]
            )

            if relevant and isinstance(idx, pd.DatetimeIndex):
                candidates.append((name, obj))

    except Exception:
        pass

print("\nTIME-SERIES DATAFRAMES FOUND:\n")

for name, df in candidates:
    print("-" * 80)
    print(f"VARIABLE: {name}")
    print(f"SHAPE:    {df.shape}")
    print(f"INDEX:    {type(df.index).__name__}")
    print(f"START:    {df.index.min()}")
    print(f"END:      {df.index.max()}")
    print(f"COLUMNS:  {list(df.columns)}")

print("\n" + "=" * 80)
print("END OF EXTRACTION")
print("=" * 80)

Q-SCANNER V7 — OOS TIME-SERIES EXTRACTION

TIME-SERIES DATAFRAMES FOUND:

--------------------------------------------------------------------------------
VARIABLE: data
SHAPE:    (974, 19)
INDEX:    DatetimeIndex
START:    2022-10-18 00:00:00
END:      2026-09-04 00:00:00
COLUMNS:  ['Close', 'High', 'Low', 'Open', 'Volume', 'SMA_20', 'SMA_50', 'SMA_200', 'RSI', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'Momentum_10', 'ATR', 'ATR_Pct', 'ADX', 'V6 Score', 'V6 Signal', 'V6 Reasons']
--------------------------------------------------------------------------------
VARIABLE: signal
SHAPE:    (974, 3)
INDEX:    DatetimeIndex
START:    2022-10-18 00:00:00
END:      2026-09-04 00:00:00
COLUMNS:  ['V6 Score', 'V6 Signal', 'V6 Reasons']
--------------------------------------------------------------------------------
VARIABLE: scanned
SHAPE:    (1124, 15)
INDEX:    DatetimeIndex
START:    2022-03-15 00:00:00
END:      2026-09-04 00:00:00
COLUMNS:  ['Close', 'High', 'Low', 'Open', 'Volume', 'SMA_20', '